# 14 Multiomic Pseudobulk RPPA Integration

## Overview
This notebook performs end-to-end integration of pseudobulk snRNA-seq and RPPA proteomics to quantify concordance, identify treatment-induced rewiring (OG vs Mock), and generate publication-ready outputs.

## Scientific Scope
- Build pathway- and TF-level activity summaries from pseudobulk RNA
- Compare RNA-informed signatures against RPPA protein measurements
- Quantify directional coupling shifts (OG - Mock) across cell types
- Prioritize robust drivers using significance-aware and effect-size-aware ranking

## Inputs and Dependencies
- Canonical mouse artifacts from notebooks `02`, `06`, and `07`
- Differential and enrichment context from notebooks `09`, `10`, and `12`
- RPPA matrix export from Notebook 13 (`RPPA_nb14_ready_2way_m0_mockog.csv`)
- Ortholog map used for cross-species pathway and TF projection

## Analysis Contract
- Harmonize features and sample identities before any cross-modal statistics
- Preserve both effect size and inferential layers (permutation/FDR and p-threshold blocks)
- Separate exploratory figures from final publication exports
- Write all final artifacts to `Results/multiomic/`

## Final Deliverables
- Master pathway and TF correlation tables
- Significance-layer summary tables (CSV/XLSX)
- Integrated pathway and TF heatmaps plus directional shift figures (SVG)
- Notebook-contained output documentation for handoff and reproducibility

# Table of Contents

1. [Overview](#overview)
2. [Scientific Scope](#scientific-scope)
3. [Inputs and Dependencies](#inputs-and-dependencies)
4. [Analysis Contract](#analysis-contract)
5. [Final Deliverables](#final-deliverables)
6. [Output Documentation](#output-documentation)
7. [Environment and Libraries](#environment-and-libraries)
8. [Pipeline Outline](#pipeline-outline)
9. [RPPA Import and Processing](#rppa-import-and-processing)

## Output Documentation

### Output Root
- `Results/multiomic/`

### Core Tables
- `Master_Cell_Pathway_RPPA_Correlations_nb14.csv`
  - Pathway-protein concordance table by `cell_type` and `group`
  - Includes effect-size style `spearman_r` and derived `delta_r`
- `Master_Cell_Pathway_RPPA_Correlations_stats.csv`
  - Pathway-level inferential layer (paired permutation statistics)
  - Includes `p_value`, `q_value`, `delta_r_pathway`, `delta_r_rms`, and paired protein counts

### TF Activity and Correlation Tables
- `Sample_Level_TF_Activities_nb14.csv`
  - Long-format sample-level DoRothEA activity table (`base_sample`, `cell_type`, `group`, `tf_name`, `activity_score`)
- `Sample_Level_TF_Activities_ByCell_nb14.xlsx`
  - Same TF activity data split into per-cell-type tabs
- `Master_TF_RPPA_Correlations_nb14.csv`
  - TF-to-RPPA concordance outputs across groups/cell types
- `Master_TF_RPPA_Correlations_ByCell_nb14.xlsx`
  - Per-cell-type workbook export for TF-RPPA concordance
- `TF_Significance_AllCells_nb14.csv`
  - TF test results across all eligible cell types (`delta`, `p_value`, `q_value`)
- `TF_Significance_AllCells_ByCell_nb14.xlsx`
  - Same TF significance results split by cell-type tabs

### Figure Outputs (SVG)
- `Sample_Pathway_Heatmap_Final_nb14.svg`
- `Pseudobulk_RNA_RPPA_HighConf_Correlation_nb14.svg`
- `Average_Pathway_RPPA_Heatmap_nb14.svg`
- `Figure_Integrated_Correlation_nb14.svg`
- `Directional_Delta_Signaling_Heatmap_nb14.svg`
- `Figure_Integrated_Directional_Shift_Correlation_nb14.svg`
- `Figure_TopDiversified_Directional_Hotspot_Heatmap_nb14.svg`
- `Figure_Neural_Specificity_Heatmap_nb14.svg`
- `Full_TF_Significance_Map_AllCells_nb14.svg`
- `Fig_Significance_Selected_TF_Pairs_nb14.svg`
- `Fig_Consolidated_Significance_Selected_nb14.svg`

### Interpretation Notes
- Delta direction convention is `OG - Mock` unless otherwise stated in-cell.
- TF full-map stars correspond to p-thresholds (`* p < 0.05`, `** p < 0.01`).
- Pathway significance stars follow the inferential layer (`q_value`) in pathway-specific panels.

### Reproducibility Notes
- Notebook assumes RPPA export from Notebook 13 and aligned sample IDs.
- Results are deterministic up to stochastic components using configured seeds where provided.
- Re-running final plotting cells overwrites same-named SVGs in `Results/multiomic/`.

## Environment and Libraries

In [ ]:
from pathlib import Path
import logging
import os
import re
import textwrap
import warnings
from typing import Any, Dict, List, Tuple

warnings.filterwarnings(
    "ignore",
    message=r"urllib3 .* or chardet.*charset_normalizer.* doesn't match a supported version!",
    category=Warning,
)

import anndata as ad
import decoupler as dc
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import torch
import torch.nn.functional as F
from graphviz import Digraph
from joblib import Parallel, delayed
from scipy import sparse
from scipy.stats import spearmanr, ttest_ind, zscore
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("notebook14")

# Output directory for all multiomic pipeline results
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "Notebooks":
    NOTEBOOK_DIR = (NOTEBOOK_DIR / "Notebooks").resolve()
ANALYSIS_DIR = NOTEBOOK_DIR.parent.resolve()
PROJECT_DIR = ANALYSIS_DIR.parent.resolve()
RESULTS_DIR = ANALYSIS_DIR / "Results" / "multiomic"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results dir: {RESULTS_DIR}")


# Pipeline Outline

## Visual Pipeline Outline

```mermaid
flowchart TD
    A[Inputs\nNB02 all-cells h5ad\nNB13 RPPA export\nOrtholog map] --> B[Environment + Config\npaths, thresholds, logging]

    B --> C[RPPA Import + Normalize\nGene-indexed matrix\nlog2 transform]
    B --> D[snRNA Ingest + Pseudobulk\nsample x cell_type groups\nbiomass filters]

    C --> E[Sample Alignment\nRNA groups matched to RPPA samples]
    D --> E

    E --> F[Pathway Activity Layer\nPROGENy human-to-mouse mapping\nMLM scoring]
    E --> G[Global RNA-RPPA Coupling\nshared feature harmonization\nSpearman correlation]

    F --> H[Pathway x Protein Correlation Table\ncell_type + group effect sizes]
    H --> I[Inferential Layer\npaired permutation tests\nBH-FDR q-values]

    E --> J[TF Activity Layer\nDoRothEA network\nULM TF scoring]
    J --> K[TF x Protein Correlation + Delta-R\nall-cell significance tables\np-threshold star maps]

    I --> L[Integrated Figures\npathway heatmaps\ndirectional shift panels]
    G --> L
    K --> L

    H --> M[Tabular Outputs\nmaster pathway correlations]
    I --> M
    J --> M
    K --> M

    L --> N[Final Deliverables\nSVG figures + CSV/XLSX tables\nResults/multiomic]
    M --> N
```

### Stage Guide
- `Inputs`: canonical snRNA data, RPPA export, and ortholog mappings.
- `Core Integration`: pseudobulk construction, RPPA normalization, and strict sample alignment.
- `Pathway Branch`: PROGENy scoring, pathway-protein effect sizes, permutation/FDR testing.
- `TF Branch`: DoRothEA TF scoring, TF-protein correlations, delta and significance summaries.
- `Outputs`: publication-ready SVGs and structured result tables in `Results/multiomic/`.

# RPPA Import


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def resolve_rppa_export() -> Path:
    """Resolve the NB13 RPPA export using the updated canonical handoff location first."""
    candidates = [
        ANALYSIS_DIR / "RPPA" / "for_multiomic" / "RPPA_nb14_ready_2way_m0_mockog.csv",
        ANALYSIS_DIR / "RPPA" / "final_simple" / "for_multiomic" / "RPPA_nb14_ready_2way_m0_mockog.csv",
        ANALYSIS_DIR / "Results" / "RPPA" / "for_multiomic" / "RPPA_nb14_ready_2way_m0_mockog.csv",
        ANALYSIS_DIR / "RPPA" / "final_simple" / "for_multiomic" / "RPPA_nb13_ready_2way_m0_mockog.csv",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    formatted = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Missing RPPA export from Notebook 13. Checked:\n" + formatted +
        "\nRun Notebook 13 export cells before executing Notebook 14."
    )


# Direct RPPA import from Notebook 13 final export (2-way, M0-only).
rppa_input_csv = resolve_rppa_export()

# File contract: Gene + Mock/OG sample columns.
df = pd.read_csv(rppa_input_csv)
if "Gene" not in df.columns:
    raise ValueError("RPPA input must include a 'Gene' column.")

# Prepare matrix exactly as downstream NB14 multiomic blocks expect.
df_rppa = df.set_index("Gene").T
df_rppa.index = df_rppa.index.astype(str).str.replace("_", "-", regex=False)
df_rppa_numeric = df_rppa.apply(pd.to_numeric, errors="coerce")
nan_cols = df_rppa_numeric.columns[df_rppa_numeric.isna().all()]
print("Columns with all NaN after conversion:", nan_cols)

# Keep legacy variable name used in downstream cells.
df_log = np.log2(df_rppa_numeric + 1)

print(f"Loaded RPPA export: {rppa_input_csv}")
print(f"Raw RPPA shape (genes x samples): {df.shape}")
print(f"NB14 RPPA matrix shape (samples x genes): {df_log.shape}")
print("Sample IDs preview:", df_log.index[:6].tolist())
print("Group counts:")
print(pd.Series(["Mock" if str(s).startswith("Mock-") else "OG" for s in df_log.index]).value_counts().to_string())

# snRNA-seq Ingest and Pseudobulk + PROGENy Integration

Loads the canonical all-cells AnnData from NB02 (`brain_allcells_allgenes.h5ad`), aggregates to pseudobulk (mean log-normalised expression per sample × cluster), then integrates with the RPPA matrix via PROGENy pathway scoring.


### Load NB02 All-Cells AnnData and Build Pseudobulk Matrix


In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse

# --- Configuration ---
ADATA_PATH        = ANALYSIS_DIR / "adatas" / "brain_allcells_allgenes.h5ad"
CELLTYPE_COL      = "class_name"     # obs column used to define pseudobulk groups
SAMPLE_COL        = "Sample"         # obs column for biological replicate
BIOMASS_THRESHOLD = 10               # min cells per (sample x celltype) group to keep

# Ingest-time statistical power filter (cell-type representation across samples).
# A cell type is retained only if it has >= MIN_CELLS_PER_SAMPLE_PER_CELLTYPE
# cells in every sample when REQUIRE_ALL_SAMPLES is True.
MIN_CELLS_PER_SAMPLE_PER_CELLTYPE = BIOMASS_THRESHOLD
REQUIRE_ALL_SAMPLES = True
MIN_SAMPLES_PER_CELLTYPE = None  # used only when REQUIRE_ALL_SAMPLES=False

# --- Load ---
if not ADATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing NB02 all-cells AnnData: {ADATA_PATH}\n"
        "Run 02_Mouse_PrepForComparison.ipynb first."
    )

adata = sc.read_h5ad(ADATA_PATH)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"  Samples ({adata.obs[SAMPLE_COL].nunique()}): {sorted(adata.obs[SAMPLE_COL].unique())}")
print(f"  Cell-type column '{CELLTYPE_COL}' - {adata.obs[CELLTYPE_COL].nunique()} unique values")

# Normalize metadata to strings once so downstream grouping is consistent.
obs_df = adata.obs[[SAMPLE_COL, CELLTYPE_COL]].copy()
obs_df[SAMPLE_COL] = obs_df[SAMPLE_COL].astype(str)
obs_df[CELLTYPE_COL] = obs_df[CELLTYPE_COL].astype(str)

if obs_df[SAMPLE_COL].str.contains("_", regex=False).any() or obs_df[CELLTYPE_COL].str.contains("_", regex=False).any():
    print("WARNING: underscores detected in sample or cell-type labels; prefer pseudobulk_meta over parsing pseudobulk index strings.")

# --- Ingest-time representation filter by cell type x sample ---
ct_sample_counts = pd.crosstab(obs_df[CELLTYPE_COL], obs_df[SAMPLE_COL])
if REQUIRE_ALL_SAMPLES:
    eligible_celltypes = ct_sample_counts.index[
        (ct_sample_counts >= MIN_CELLS_PER_SAMPLE_PER_CELLTYPE).all(axis=1)
    ]
    min_samples_required = ct_sample_counts.shape[1]
else:
    min_samples_required = (
        MIN_SAMPLES_PER_CELLTYPE
        if MIN_SAMPLES_PER_CELLTYPE is not None
        else ct_sample_counts.shape[1]
    )
    eligible_celltypes = ct_sample_counts.index[
        (ct_sample_counts >= MIN_CELLS_PER_SAMPLE_PER_CELLTYPE).sum(axis=1) >= min_samples_required
    ]

n_ct_before = adata.obs[CELLTYPE_COL].nunique()
adata = adata[adata.obs[CELLTYPE_COL].isin(eligible_celltypes)].copy()
n_ct_after = adata.obs[CELLTYPE_COL].nunique()
print(
    f"Representation filter: retained {n_ct_after}/{n_ct_before} cell types "
    f"(min {MIN_CELLS_PER_SAMPLE_PER_CELLTYPE} cells in {min_samples_required} samples)"
)

# Rebuild normalized metadata after filtering to keep row alignment exact.
obs_df = adata.obs[[SAMPLE_COL, CELLTYPE_COL]].copy()
obs_df[SAMPLE_COL] = obs_df[SAMPLE_COL].astype(str)
obs_df[CELLTYPE_COL] = obs_df[CELLTYPE_COL].astype(str)

# --- Choose expression matrix ---
# NB02 stores log1p-normalised counts in layers['log_norm']; use that to avoid
# double-normalising. Fall back to raw X with a warning if absent.
if "log_norm" in adata.layers:
    X = adata.layers["log_norm"]
    print("Using layers['log_norm'] (log1p-normalised from NB02).")
else:
    print("WARNING: 'log_norm' layer not found - using raw adata.X. Consider re-running NB02.")
    X = adata.X

# --- Pseudobulk aggregation (mean per sample x celltype) ---
# Sparse indicator matrix avoids converting the full X to dense.
obs_df["group_key"] = obs_df[SAMPLE_COL] + "_" + obs_df[CELLTYPE_COL]
obs_df["sample"] = obs_df[SAMPLE_COL]
obs_df["cell_type"] = obs_df[CELLTYPE_COL]

group_cat  = pd.Categorical(obs_df["group_key"])
n_cells    = X.shape[0]
n_groups   = len(group_cat.categories)

indicator  = scipy.sparse.csr_matrix(
    (np.ones(n_cells), (np.arange(n_cells), group_cat.codes)),
    shape=(n_cells, n_groups),
)

group_counts = np.array(indicator.sum(axis=0)).ravel()
if (group_counts == 0).any():
    raise ValueError("Encountered empty pseudobulk groups; grouping metadata is inconsistent with the indicator matrix.")

if scipy.sparse.issparse(X):
    pb_sum = (indicator.T @ X).toarray()
else:
    pb_sum = indicator.T @ np.asarray(X)

pb_mean = pb_sum / group_counts[:, np.newaxis]

pseudobulk = pd.DataFrame(
    pb_mean,
    index=group_cat.categories,
    columns=adata.var_names,
)

pseudobulk_meta = (
    obs_df[["group_key", "sample", "cell_type"]]
    .drop_duplicates()
    .set_index("group_key")
    .reindex(group_cat.categories)
)
pseudobulk_meta["n_cells"] = group_counts

# --- Biomass filter on sample x celltype groups ---
cell_count_map = pd.Series(group_counts, index=group_cat.categories)
keep           = cell_count_map[cell_count_map >= BIOMASS_THRESHOLD].index
n_before       = len(pseudobulk)
pseudobulk     = pseudobulk.loc[keep]
pseudobulk_meta = pseudobulk_meta.loc[keep].copy()

print(f"\nPseudobulk shape (samples x genes): {pseudobulk.shape}")
print(f"Biomass filter (>={BIOMASS_THRESHOLD} cells): {n_before} -> {len(pseudobulk)} groups retained")
print(f"Samples in pseudobulk: {sorted(pseudobulk_meta['sample'].unique())}")
print(f"Final pseudobulk shape: {pseudobulk.shape}")

### Master Pathway Correlation Table

In [ ]:
# --- 0. Configuration ---
PIPELINE_LOG_PATH = RESULTS_DIR / "pipeline_log_correlations.txt"
PROGENY_HUMAN_TOP = 500
PROGENY_TARGETS_PER_PATHWAY = 100
MIN_RPPA_REPS_PER_GROUP = 3
N_JOBS = max(1, min(30, (os.cpu_count() or 1) - 1))
PREFERRED_DELTA_COMPARISON = ("OG", "Mock")
ORTHOLOG_PATH = Path(
    str(ANALYSIS_DIR / "Microglia_analysis" / "mouse_human_orthologs_nb02.csv")
)

# --- 0.5 Logging Configuration ---
logger = logging.getLogger("notebook14.pathway_corr")
logger.setLevel(logging.INFO)
logger.handlers = []
logger.propagate = False
_fmt = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
_fh = logging.FileHandler(PIPELINE_LOG_PATH)
_fh.setFormatter(_fmt)
_sh = logging.StreamHandler()
_sh.setFormatter(_fmt)
logger.addHandler(_fh)
logger.addHandler(_sh)

logger.info("Aligning pseudobulk with RPPA samples and preparing matrices...")

# --- 1. Metadata-safe alignment (prefer pseudobulk_meta, fallback to index split) ---
pb_expr = pseudobulk.copy()
if "pseudobulk_meta" in globals() and isinstance(pseudobulk_meta, pd.DataFrame):
    pb_meta_local = pseudobulk_meta.reindex(pb_expr.index).copy()
    required_cols = {"sample", "cell_type"}
    missing_cols = required_cols - set(pb_meta_local.columns)
    if missing_cols:
        raise ValueError(f"pseudobulk_meta missing required columns: {missing_cols}")
else:
    logger.warning("pseudobulk_meta not found; falling back to index parsing. Consider re-running Cell 11.")
    parsed = pb_expr.index.to_series().astype(str).str.split("_", n=1, expand=True)
    if parsed.shape[1] != 2:
        raise ValueError("Could not parse pseudobulk index into sample/cell_type with '_' separator.")
    pb_meta_local = pd.DataFrame(index=pb_expr.index)
    pb_meta_local["sample"] = parsed[0].values
    pb_meta_local["cell_type"] = parsed[1].values

pb_meta_local["sample"] = pb_meta_local["sample"].astype(str)
pb_meta_local["cell_type"] = pb_meta_local["cell_type"].astype(str)

if pb_meta_local[["sample", "cell_type"]].isna().any().any():
    raise ValueError("Detected missing sample/cell_type metadata after alignment.")

aligned_mask = pb_meta_local["sample"].isin(df_log.index)
n_before_align = int(pb_expr.shape[0])
pb_expr = pb_expr.loc[aligned_mask].copy()
pb_meta_local = pb_meta_local.loc[aligned_mask].copy()
logger.info(
    f"Alignment complete: retained {pb_expr.shape[0]}/{n_before_align} pseudobulk groups with RPPA-matched samples."
)

if pb_expr.empty:
    raise ValueError("No pseudobulk groups remain after RPPA sample alignment.")

expr_pb = pb_expr
shared_proteins = df_log.select_dtypes(include=[np.number]).columns.tolist()
if len(shared_proteins) == 0:
    raise ValueError("No numeric RPPA protein columns available in df_log.")

def infer_group_label(sample_id: str) -> str:
    """Infer experimental group from sample id without assuming only one delimiter style."""
    s = str(sample_id).strip()
    su = s.upper()
    if "MOCK" in su:
        return "Mock"
    if su.startswith("OG") or re.search(r"(^|[-_])OG($|[-_])", su):
        return "OG"
    return np.nan

rppa_group_map = pd.Series(
    [infer_group_label(idx) for idx in df_log.index],
    index=df_log.index,
    name="group",
)
unknown_rppa = int(rppa_group_map.isna().sum())
if unknown_rppa > 0:
    logger.warning(f"{unknown_rppa} RPPA samples have unknown group labels and will be ignored in group benchmarks.")

def compute_ct_correlations(
    ct: str,
    group: str,
    sub_df: pd.DataFrame,
    rppa_df: pd.DataFrame,
    prot_list: list,
    path_list: list,
    rppa_groups: pd.Series,
    min_reps: int = 3,
    ) -> List[Dict[str, Any]]:
    """Compute pathway x protein rank-vector concordance per (cell_type, group)."""
    group_indices = rppa_groups[rppa_groups == group].index.tolist()
    if len(group_indices) < min_reps or sub_df.empty or len(path_list) == 0:
        return []

    prot_bench = rppa_df.loc[group_indices, prot_list].mean(axis=0)
    rna_bench = sub_df[path_list].mean(axis=0)

    r_prot = prot_bench.rank().values
    r_rna = rna_bench.rank().values

    a = r_rna - r_rna.mean()
    b = r_prot - r_prot.mean()
    den = np.linalg.norm(a) * np.linalg.norm(b)
    if den <= 1e-12:
        return []

    r_matrix = np.outer(a, b) / den

    ct_results = []
    for i, path in enumerate(path_list):
        for j, prot in enumerate(prot_list):
            ct_results.append(
                {
                    "cell_type": ct,
                    "group": group,
                    "pathway": path,
                    "protein": prot,
                    "feature": f"{path} | {prot}",
                    "spearman_r": float(r_matrix[i, j]),
                    "p_value": np.nan,
                    "n_samples": int(len(sub_df)),
                    "n_rppa_reps": int(len(group_indices)),
                    "metric_note": "rank-vector concordance (effect size, non-inferential)",
                }
            )
    return ct_results

# --- 2. Compute Pathway Activities (PROGENy human->mouse via local ortholog map) ---
logger.info("Initializing PROGENy model and computing pathway scores...")
progeny_human = dc.op.progeny(organism="human", top=PROGENY_HUMAN_TOP)
orthologs = pd.read_csv(ORTHOLOG_PATH)
h2m = dict(zip(orthologs["Human_Gene"], orthologs["Mouse_Gene"]))
progeny_net = progeny_human.copy()
progeny_net["target"] = progeny_net["target"].map(h2m)
progeny_net = (
    progeny_net.dropna(subset=["target"])
    .sort_values("padj")
    .groupby("source")
    .head(PROGENY_TARGETS_PER_PATHWAY)
    .reset_index(drop=True)
)
logger.info(
    f"PROGENy mouse network: {len(progeny_net)} gene-pathway pairs across {progeny_net['source'].nunique()} pathways"
)

pathway_scores, _ = dc.mt.mlm(data=expr_pb, net=progeny_net)
pathway_cols = pathway_scores.columns.tolist()

# --- 3. Metadata integration ---
pathway_scores["base_sample"] = pb_meta_local["sample"].values
pathway_scores["cell_type"] = pb_meta_local["cell_type"].values
pathway_scores["group"] = pathway_scores["base_sample"].map(infer_group_label)

unknown_groups = int(pathway_scores["group"].isna().sum())
if unknown_groups > 0:
    logger.warning(f"Dropping {unknown_groups} pathway rows with unknown group labels.")
pathway_scores = pathway_scores.dropna(subset=["group"]).copy()

# --- 4. Parallel execution by (cell_type, group) ---
group_tasks = list(pathway_scores.groupby(["cell_type", "group"], observed=True))
logger.info(f"Starting parallel correlation on {N_JOBS} cores for {len(group_tasks)} unique groups.")

results_lists = Parallel(n_jobs=N_JOBS)(
    delayed(compute_ct_correlations)(
        ct, grp, sub, df_log, shared_proteins, pathway_cols, rppa_group_map, MIN_RPPA_REPS_PER_GROUP
    )
    for (ct, grp), sub in tqdm(group_tasks, desc="Processing Groups")
)

corr_table = pd.DataFrame([item for sublist in results_lists for item in sublist])

# --- 5. Delta effect-size post-processing ---
if not corr_table.empty:
    logger.info("Performing Delta R calculation...")
    pivot_r = corr_table.pivot_table(
        index=["cell_type", "feature"],
        columns="group",
        values="spearman_r",
    ).reset_index()

    g_pos, g_ref = PREFERRED_DELTA_COMPARISON
    if g_pos in pivot_r.columns and g_ref in pivot_r.columns:
        pivot_r["delta_r"] = pivot_r[g_pos] - pivot_r[g_ref]
        pivot_r["delta_comparison"] = f"{g_pos}-{g_ref}"
    else:
        available = [c for c in pivot_r.columns if c not in ["cell_type", "feature"]]
        if len(available) >= 2:
            g1, g2 = sorted(available)[:2]
            pivot_r["delta_r"] = pivot_r[g1] - pivot_r[g2]
            pivot_r["delta_comparison"] = f"{g1}-{g2}"
            logger.warning(
                f"Preferred delta groups not fully available; using fallback comparison {g1}-{g2}."
            )
        else:
            pivot_r["delta_r"] = np.nan
            pivot_r["delta_comparison"] = np.nan
            logger.warning("Insufficient groups to compute delta_r.")

    corr_table = corr_table.merge(
        pivot_r[["cell_type", "feature", "delta_r", "delta_comparison"]],
        on=["cell_type", "feature"],
        how="left",
    )
else:
    logger.error("Correlation table is empty. Check sample alignment, group labels, and RPPA index.")

# --- 6. Export ---
output_path = RESULTS_DIR / "Master_Cell_Pathway_RPPA_Correlations_nb14.csv"
corr_table.to_csv(output_path, index=False)
logger.info(f"Pipeline complete. Results saved to {output_path}")

### Pathway Significance Layer (Permutation + FDR)

This block keeps the master effect-size table unchanged and adds inferential statistics at the pathway x cell-type level.

- Test unit: paired protein-level deltas (OG - Mock) within each pathway/cell-type
- Null model: random sign-flips of paired protein deltas
- Multiple testing: Benjamini-Hochberg FDR across tested pathway/cell-type pairs

In [ ]:
# =========================================================
# Significance Layer: pathway-level paired permutation tests
# =========================================================
N_PERMUTATIONS = 5000
MIN_PROTEINS_PER_TEST = 20
RNG_SEED = 42

def pairing_permutation_pvalue_rms(
    og_vals: np.ndarray, mock_vals: np.ndarray, n_perm: int = 5000, seed: int = 42
    ) -> float:
    """Permutation p-value for RMS(OG - Mock) under random protein pairing null."""
    og_vals = np.asarray(og_vals, dtype=float)
    mock_vals = np.asarray(mock_vals, dtype=float)

    valid = np.isfinite(og_vals) & np.isfinite(mock_vals)
    og_vals = og_vals[valid]
    mock_vals = mock_vals[valid]
    n = og_vals.size
    if n < 3:
        return np.nan

    obs = float(np.sqrt(np.mean((og_vals - mock_vals) ** 2)))
    rng = np.random.default_rng(seed)

    perm_stats = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        perm_mock = mock_vals[rng.permutation(n)]
        perm_stats[i] = np.sqrt(np.mean((og_vals - perm_mock) ** 2))

    # Upper-tail: larger RMS indicates stronger OG-vs-Mock mismatch than random pairing.
    p = (np.sum(perm_stats >= obs) + 1.0) / (n_perm + 1.0)
    return float(p)

required_cols = {"cell_type", "group", "pathway", "protein", "spearman_r"}
missing = required_cols - set(corr_table.columns)
if missing:
    raise ValueError(f"corr_table missing required columns for stats layer: {missing}")

pairs = []
for (ct, pathway), sub in corr_table.groupby(["cell_type", "pathway"], observed=True):
    og = sub[sub["group"] == "OG"][ ["protein", "spearman_r"] ].dropna()
    mock = sub[sub["group"] == "Mock"][ ["protein", "spearman_r"] ].dropna()

    if og.empty or mock.empty:
        continue

    merged = og.merge(mock, on="protein", suffixes=("_og", "_mock"))
    if merged.shape[0] < MIN_PROTEINS_PER_TEST:
        continue

    og_vals = merged["spearman_r_og"].values
    mock_vals = merged["spearman_r_mock"].values
    diff = og_vals - mock_vals

    mean_delta = float(np.mean(diff))
    rms_delta = float(np.sqrt(np.mean(diff ** 2)))
    p_val = pairing_permutation_pvalue_rms(
        og_vals, mock_vals, n_perm=N_PERMUTATIONS, seed=RNG_SEED
    )

    pairs.append(
        {
            "cell_type": ct,
            "pathway": pathway,
            "n_paired_proteins": int(merged.shape[0]),
            "mock_mean_r": float(np.mean(mock_vals)),
            "og_mean_r": float(np.mean(og_vals)),
            "delta_r_pathway": mean_delta,
            "delta_r_rms": rms_delta,
            "p_value": p_val,
            "test_method": "protein_pairing_permutation_rms_delta",
            "n_permutations": int(N_PERMUTATIONS),
            "min_proteins_required": int(MIN_PROTEINS_PER_TEST),
        }
    )

pathway_stats = pd.DataFrame(pairs)
if pathway_stats.empty:
    logger.warning("No pathway-level tests were computed (insufficient paired proteins per test).")
else:
    pathway_stats["q_value"] = np.nan
    valid = pathway_stats["p_value"].notna()
    if valid.any():
        pathway_stats.loc[valid, "q_value"] = multipletests(
            pathway_stats.loc[valid, "p_value"], method="fdr_bh"
        )[1]

    pathway_stats = pathway_stats.sort_values(["q_value", "p_value", "delta_r_rms"], na_position="last")

    stats_out = RESULTS_DIR / "Master_Cell_Pathway_RPPA_Correlations_stats.csv"
    pathway_stats.to_csv(stats_out, index=False)
    logger.info(f"Significance layer complete. Stats saved to {stats_out}")
    print(f"Computed pathway-level tests: {len(pathway_stats)}")
    print(f"FDR-significant (q < 0.10): {(pathway_stats['q_value'] < 0.10).sum()}")
    print(f"Nominally significant (p < 0.05): {(pathway_stats['p_value'] < 0.05).sum()}")

# Optional join for convenience in downstream plotting blocks
if not pathway_stats.empty:
    corr_table_stats = corr_table.merge(
        pathway_stats[["cell_type", "pathway", "delta_r_pathway", "delta_r_rms", "p_value", "q_value", "n_paired_proteins"]],
        on=["cell_type", "pathway"],
        how="left",
    )
else:
    corr_table_stats = corr_table.copy()

# Pseudobulk Pathway Correlation

## Samples to Pathway Correlation

#### corr_SamplePath

In [ ]:
# =========================================================
# 1. BRIDGE: Create corr_SamplePath from Pipeline Outputs
# =========================================================
logger.info("Generating sample-level pathway activity table for heatmap visualization...")

required_cols = {'base_sample', 'group', *pathway_cols}
missing_cols = required_cols.difference(pathway_scores.columns)
if missing_cols:
    raise KeyError(f"pathway_scores is missing required columns: {sorted(missing_cols)}")

sample_group_check = (
    pathway_scores[['base_sample', 'group']]
    .dropna()
    .drop_duplicates()
    .groupby('base_sample', observed=True)['group']
    .nunique()
)

if (sample_group_check > 1).any():
    bad_samples = sample_group_check[sample_group_check > 1].index.tolist()
    raise ValueError(f"Conflicting group assignments detected for samples: {bad_samples}")

sample_level_scores = (
    pathway_scores[['base_sample', 'group', *pathway_cols]]
    .dropna(subset=['base_sample', 'group'])
    .groupby(['base_sample', 'group'], observed=True)[pathway_cols]
    .mean()
    .reset_index()
)

if sample_level_scores.empty:
    raise ValueError('No sample-level pathway scores available for heatmap generation.')

corr_SamplePath = sample_level_scores.melt(
    id_vars=['base_sample', 'group'],
    value_vars=pathway_cols,
    var_name='pathway',
    value_name='activity_score',
).rename(columns={'base_sample': 'sample_id'})

corr_SamplePath['group'] = pd.Categorical(
    corr_SamplePath['group'],
    categories=['Mock', 'OG'],
    ordered=True,
)

missing_activity = int(corr_SamplePath['activity_score'].isna().sum())
if missing_activity > 0:
    logger.warning(f"Dropping {missing_activity} sample-pathway rows with missing activity scores.")
    corr_SamplePath = corr_SamplePath.dropna(subset=['activity_score']).copy()

# --- QC: confirm sample-level heterogeneity was preserved after aggregation ---
avg_std = corr_SamplePath.groupby('pathway', observed=True)['activity_score'].std().mean()
logger.info(
    f"Prepared {corr_SamplePath['sample_id'].nunique()} samples across "
    f"{corr_SamplePath['pathway'].nunique()} pathways. Average pathway StdDev: {avg_std:.4f}"
)
if not np.isfinite(avg_std) or avg_std <= 0:
    raise ValueError('No sample-level pathway variance detected after aggregation. Check pathway_scores inputs.')

### Figure Sample Pseudobulk Pathway Correlation Heatmap

In [ ]:
# =========================================================
# 2. PLOT: Sample vs Pathway Heatmap
# =========================================================
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap

sample_meta = corr_SamplePath[['sample_id', 'group']].drop_duplicates().copy()
if sample_meta['sample_id'].duplicated().any():
    raise ValueError('corr_SamplePath contains duplicate sample_id to group mappings.')

sample_meta['group'] = pd.Categorical(sample_meta['group'], categories=['Mock', 'OG'], ordered=True)
sample_meta = sample_meta.sort_values(['group', 'sample_id'])
sorted_cols = sample_meta['sample_id'].tolist()

heat_matrix = corr_SamplePath.pivot_table(
    index='pathway',
    columns='sample_id',
    values='activity_score',
    aggfunc='mean',
)
heat_matrix = heat_matrix.reindex(columns=sorted_cols)

if heat_matrix.empty:
    raise ValueError('No sample-pathway matrix available for heatmap plotting.')

# Scale by row (Z-score) while protecting against zero-variance pathways.
row_stds = heat_matrix.std(axis=1).replace(0, np.nan)
heat_matrix_z = heat_matrix.sub(heat_matrix.mean(axis=1), axis=0).div(row_stds, axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(14, 10))

sns.heatmap(
    heat_matrix_z,
    cmap='vlag',
    center=0,
    robust=True,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Relative Activity (Z-score)'},
    ax=ax,
)

mock_count = int((sample_meta['group'] == 'Mock').sum())
if mock_count > 0 and mock_count < len(sorted_cols):
    ax.axvline(mock_count, color='black', lw=4, linestyle='-')

ax.set_title('Replicate-Level Signaling Heterogeneity', fontsize=22, fontweight='bold', pad=50)
ax.set_ylabel('PROGENy Pathway', fontsize=16, fontweight='bold')
ax.set_xlabel('Biological Replicates', fontsize=16, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=12)

if mock_count > 0:
    plt.text(mock_count / 2, -0.6, 'MOCK', ha='center', fontsize=15, fontweight='bold', color='blue')
if mock_count < len(sorted_cols):
    plt.text(mock_count + (len(sorted_cols) - mock_count) / 2, -0.6, 'OG', ha='center', fontsize=15, fontweight='bold', color='red')

plt.tight_layout()
save_path = RESULTS_DIR / 'Sample_Pathway_Heatmap_Final_nb14.svg'
plt.savefig(save_path, format='svg', bbox_inches='tight')
print(f'Saved sample-pathway heatmap to {save_path}')
plt.show()

## Cells to Pathways Correlation

In [ ]:
# --- Pathway summary heatmap inputs: significance-driven with effect-size fallback ---
# Pathways are selected by significance (q < SIG_Q_CUTOFF) when available.
# All cell types are shown and ordered alphabetically.
SIG_Q_CUTOFF = 0.05
MAX_PATHWAYS = 15

stats_df = corr_table_stats.copy() if 'corr_table_stats' in globals() else None
if stats_df is None or stats_df.empty:
    raise RuntimeError('corr_table_stats is required before building pathway summary heatmaps.')

stats_df['q_value'] = pd.to_numeric(stats_df['q_value'], errors='coerce')
stats_df['delta_r_rms'] = pd.to_numeric(stats_df['delta_r_rms'], errors='coerce')
stats_df['delta_r_pathway'] = pd.to_numeric(stats_df['delta_r_pathway'], errors='coerce')

# Always include all available cell types, alphabetically ordered
selected_celltypes = sorted(stats_df['cell_type'].dropna().unique().tolist())

sig_stats = stats_df.loc[stats_df['q_value'].lt(SIG_Q_CUTOFF)].copy()

if not sig_stats.empty:
    # Pathways ranked over all significant tests
    pathway_rank = (
        sig_stats
        .groupby('pathway')
        .agg(
            sig_celltypes=('cell_type', 'nunique'),
            best_q=('q_value', 'min'),
            max_delta_r_rms=('delta_r_rms', 'max'),
        )
        .sort_values(
            ['sig_celltypes', 'max_delta_r_rms', 'best_q'],
            ascending=[False, False, True],
        )
    )
    top_pathways = pathway_rank.head(MAX_PATHWAYS).index.tolist()
    selection_mode = f'all cell types (alphabetical); pathways significance-driven (q < {SIG_Q_CUTOFF:.2f})'
else:
    # Fallback: no significant tests — rank pathways by effect size
    pathway_rank = (
        stats_df
        .groupby('pathway')
        .agg(
            max_delta_r_rms=('delta_r_rms', 'max'),
            mean_abs_delta=('delta_r_pathway', lambda s: s.abs().mean()),
        )
        .sort_values(['max_delta_r_rms', 'mean_abs_delta'], ascending=[False, False])
    )
    top_pathways = pathway_rank.head(MAX_PATHWAYS).index.tolist()
    selection_mode = 'effect-size fallback (all cell types, alphabetical)'

subset_df = stats_df[
    stats_df['cell_type'].isin(selected_celltypes) &
    stats_df['pathway'].isin(top_pathways)
].copy()

split_heat = subset_df.pivot_table(
    index='pathway',
    columns='cell_type',
    values='delta_r_rms',
    aggfunc='mean',
)

star_annot = subset_df.assign(
    stars=subset_df['q_value'].apply(
        lambda q: '***' if pd.notna(q) and q < 0.001 else ('**' if pd.notna(q) and q < 0.01 else ('*' if pd.notna(q) and q < SIG_Q_CUTOFF else ''))
    )
).pivot_table(
    index='pathway',
    columns='cell_type',
    values='stars',
    aggfunc='first',
)

split_heat = split_heat.reindex(index=top_pathways, columns=selected_celltypes)
star_annot = star_annot.reindex(index=top_pathways, columns=selected_celltypes).fillna('')

# Keep pathway rows with signal; keep all cell-type columns to preserve alphabetical layout
split_heat = split_heat.dropna(how='all')
star_annot = star_annot.reindex(index=split_heat.index, columns=split_heat.columns).fillna('')
selected_celltypes = split_heat.columns.tolist()

if split_heat.empty:
    raise RuntimeError('No pathway shift values were available after significance-aware filtering.')

max_val = split_heat.abs().max().max()
n_sig_ct = len(set(sig_stats['cell_type'])) if not sig_stats.empty else 0
print(f'Selection mode: {selection_mode}')
print(f'Cell types in heatmap: {split_heat.shape[1]} total (alphabetical order; {n_sig_ct} with >=1 significant test)')
print('Selected pathways:', ', '.join(split_heat.index.tolist()))
print(f'Data prepared: {split_heat.shape[0]} pathways x {split_heat.shape[1]} cell types.')
print(f'Maximum absolute pathway shift in this set: {max_val:.4f}')

### Global RNA-RPPA Correlation/Decoupling

Question: how much the RNA looks like the Protein?

In [ ]:
# =========================================
# 1. Prepare RNA metadata & Align
# =========================================
# Re-building metadata from the filtered pseudobulk object
pb_meta = pd.DataFrame(index=pseudobulk.index)
pb_meta["sample_celltype"] = pb_meta.index
pb_meta["sample"] = pb_meta["sample_celltype"].str.split("_").str[0]
pb_meta["cell_type"] = pb_meta["sample_celltype"].str.split("_").str[1]

# Derive sample->group map from validated upstream metadata (no string heuristics)
_sg = (
    pathway_scores[['base_sample', 'group']]
    .dropna()
    .drop_duplicates('base_sample')
    .set_index('base_sample')['group']
)
if _sg.index.duplicated().any():
    raise ValueError('Conflicting group assignments in pathway_scores base_sample index.')
sample_to_group = _sg.to_dict()

pb_meta["group"] = pb_meta["sample"].map(sample_to_group)
pb_meta = pb_meta.dropna(subset=["group"])
pseudobulk_aligned = pseudobulk.loc[pb_meta.index]

# =========================================
# 2. Prepare RPPA Data
# =========================================
rppa_numeric = df_log.select_dtypes(include=np.number)
rppa_group_vals = pd.Series(rppa_numeric.index).map(sample_to_group)
if rppa_group_vals.isna().all():
    raise ValueError(
        'No RPPA samples found in sample_to_group map. '
        'Check that RPPA sample IDs match pathway_scores base_sample values.'
    )
rppa_group_map = pd.Series(rppa_group_vals.values, index=rppa_numeric.index)

# =========================================
# 3. Harmonize feature names and compute correlations
# =========================================
def _norm_symbol(x):
    # Case-insensitive gene/protein symbol normalization.
    return re.sub(r"[^A-Za-z0-9]", "", str(x)).upper()

pb_key_to_gene = {}
for g in pseudobulk_aligned.columns:
    k = _norm_symbol(g)
    if k and k not in pb_key_to_gene:
        pb_key_to_gene[k] = g

rppa_key_to_feat = {}
for g in rppa_numeric.columns:
    k = _norm_symbol(g)
    if k and k not in rppa_key_to_feat:
        rppa_key_to_feat[k] = g

shared_keys = sorted(set(pb_key_to_gene) & set(rppa_key_to_feat))
shared_pairs = [(pb_key_to_gene[k], rppa_key_to_feat[k]) for k in shared_keys]

# Optional fallback: expand overlaps through human->mouse ortholog map if overlap is too small.
if len(shared_pairs) < 50:
    try:
        orth = pd.read_csv(ORTHOLOG_PATH)
        h2m = {
            _norm_symbol(h): _norm_symbol(m)
            for h, m in zip(orth["Human_Gene"], orth["Mouse_Gene"])
            if pd.notna(h) and pd.notna(m)
        }
        existing = set(shared_pairs)
        for rppa_feat in rppa_numeric.columns:
            hkey = _norm_symbol(rppa_feat)
            mkey = h2m.get(hkey)
            if mkey and mkey in pb_key_to_gene:
                pair = (pb_key_to_gene[mkey], rppa_feat)
                if pair not in existing:
                    shared_pairs.append(pair)
                    existing.add(pair)
    except Exception as e:
        print(f"Ortholog fallback skipped: {e}")

shared_features = shared_pairs
print(f"Number of shared features for multi-omic integration: {len(shared_features)}")

if len(shared_features) < 10:
    raise ValueError(
        "Too few shared RNA/RPPA features after harmonization. "
        "Check symbol mapping and ortholog table."
    )

pb_features = [p[0] for p in shared_features]
rppa_features = [p[1] for p in shared_features]

corr_list = []
for _, row in pb_meta.iterrows():
    rna_sample_id = row["sample_celltype"]
    cell_type = row["cell_type"]
    group_label = row["group"]

    # RNA vector preparation
    rna_vec = pseudobulk_aligned.loc[rna_sample_id, pb_features].astype(float).values
    rna_z = zscore(rna_vec, nan_policy="omit")

    # Map to corresponding RPPA group replicates
    relevant_rppa_samples = rppa_group_map[rppa_group_map == group_label].index

    group_corrs = []
    for rppa_s in relevant_rppa_samples:
        rppa_vec = rppa_numeric.loc[rppa_s, rppa_features].astype(float).values
        rppa_z = zscore(rppa_vec, nan_policy="omit")

        valid = np.isfinite(rna_z) & np.isfinite(rppa_z)
        if valid.sum() < 3:
            continue

        # Spearman correlation (rank-based, robust to outliers)
        corr_val, _ = spearmanr(rna_z[valid], rppa_z[valid], nan_policy="omit")
        if np.isfinite(corr_val):
            group_corrs.append(float(corr_val))

    # Average correlation of this RNA sample against all group-matched RPPA replicates
    avg_corr = np.nan if len(group_corrs) == 0 else float(np.mean(group_corrs))

    corr_list.append(
        {
            "rna_sample": rna_sample_id,
            "cell_type": cell_type,
            "group": group_label,
            "correlation": avg_corr,
        }
    )
# =========================================
corr_df = pd.DataFrame(corr_list)
corr_df = corr_df[np.isfinite(corr_df["correlation"])].copy()

if corr_df.empty:
    raise ValueError("No finite correlations were computed after feature harmonization.")

# =========================================
# 4. Significance & Plotting
# =========================================
plt.figure(figsize=(16, 8))  # Slightly wider for better label spacing

# Main plot
ax = sns.boxplot(
    x="cell_type",
    y="correlation",
    hue="group",
    data=corr_df,
    palette="Set2",
    showfliers=False,
)

sns.stripplot(
    x="cell_type",
    y="correlation",
    hue="group",
    data=corr_df,
    dodge=True,
    color="0.3",
    alpha=0.8,
    legend=False,
)

# Legend clean up
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles[0:2], labels=labels[0:2], title="Condition", loc="upper right")

# Axis formatting
ax.grid(False)
ax.tick_params(axis="x", which="both", bottom=True)

# Standardized xtick rotation
unique_cts = sorted(corr_df["cell_type"].unique())
ax.set_xticks(range(len(unique_cts)))
ax.set_xticklabels(unique_cts, rotation=45, ha="right")

# Significance Assessment (Welch's T-test)
for i, ct in enumerate(unique_cts):
    m_vals = corr_df[(corr_df["cell_type"] == ct) & (corr_df["group"] == "Mock")]["correlation"]
    o_vals = corr_df[(corr_df["cell_type"] == ct) & (corr_df["group"] == "OG")]["correlation"]

    if len(m_vals) > 1 and len(o_vals) > 1:
        # equal_var=False performs Welch's T-test
        _, pval = ttest_ind(m_vals, o_vals, equal_var=False)

        if pval < 0.05:
            y_max = corr_df[corr_df["cell_type"] == ct]["correlation"].max()
            stars = "****" if pval < 0.0001 else "***" if pval < 0.001 else "**" if pval < 0.01 else "*"
            # Place stars with a small offset for visibility
            plt.text(i, y_max + 0.015, stars, ha="center", color="red", fontweight="bold", fontsize=12)

plt.title(f"RNA-Protein Regulatory Coupling\n(Strict Biomass Filter: N >= {BIOMASS_THRESHOLD} cells/group)")
plt.ylabel("Spearman Correlation (R)")
plt.xlabel("Anatomical Cell Population")
_y_top = corr_df['correlation'].max()
_y_bot = max(0.0, corr_df['correlation'].min() - 0.05)
plt.ylim(_y_bot, _y_top + 0.10)  # data-driven limits with headroom for stars

# Corrected Figure Description for Publication
plt.figtext(
    0.02,
    -0.08,
    f"Figure. Multi-omic regulatory coupling analysis. Bars represent mean Spearman correlation between "
    f"z-scored RNA pseudobulk (snRNA-seq) and RPPA protein profiles across {len(shared_features)} shared features. "
    f"Only clusters meeting a stringent biomass threshold (N >= {BIOMASS_THRESHOLD} cells per experimental group) "
    "were included. Points represent individual RNA samples. P-values determined via Welch's T-test "
    "(*p<0.05, **p<0.01, ***p<0.001).",
    wrap=True,
    horizontalalignment="left",
    fontsize=16,
)

plt.tight_layout()

# Save as SVG (Publication quality)
output_path = RESULTS_DIR / 'Pseudobulk_RNA_RPPA_HighConf_Correlation_nb14.svg'
plt.savefig(output_path, format="svg", bbox_inches="tight")

plt.show()


### Average Pathway–RPPA-Celltype Correlation

In [ ]:
# ---------------------------------------------------------
# 1. Plot pathway-level shift heatmap from stats layer
# ---------------------------------------------------------
if 'split_heat' not in globals() or split_heat.empty:
    raise RuntimeError('split_heat must be prepared before plotting the pathway summary heatmap.')
if 'star_annot' not in globals():
    raise RuntimeError('star_annot must be prepared before plotting the pathway summary heatmap.')

# Raw matrix (true effect sizes)
plot_matrix_raw = split_heat.copy()
plot_z_matrix = star_annot.reindex(index=plot_matrix_raw.index, columns=plot_matrix_raw.columns).fillna('')

# Visual-only stretch to [-1, 1] so subtle differences are easier to see
_raw_vals = plot_matrix_raw.to_numpy(dtype=float)
if not np.isfinite(_raw_vals).any():
    raise RuntimeError('No finite values found in split_heat for plotting.')

_raw_min = float(np.nanmin(_raw_vals))
_raw_max = float(np.nanmax(_raw_vals))
if _raw_max > _raw_min:
    plot_matrix = ((plot_matrix_raw - _raw_min) / (_raw_max - _raw_min)) * 2.0 - 1.0
    scale_note = f'display-normalized from raw range [{_raw_min:.4f}, {_raw_max:.4f}] to [-1, 1]'
else:
    plot_matrix = pd.DataFrame(0.0, index=plot_matrix_raw.index, columns=plot_matrix_raw.columns)
    scale_note = f'raw range is constant at {_raw_min:.4f}; plotted as 0.0'

_n_cols = plot_matrix.shape[1]
fig, ax = plt.subplots(figsize=(max(14, _n_cols), 8))

sns.heatmap(
    plot_matrix,
    cmap='vlag',
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    linecolor='white',
    annot=plot_z_matrix,
    fmt='',
    annot_kws={'fontsize': max(6, 10 - _n_cols // 5)},
    cbar_kws={'label': 'Display-normalized pathway shift (scaled to [-1, 1])'},
    ax=ax,
)

ax.set_title('Pathway-Level RNA-RPPA Shift by Cell Type', fontsize=15)
ax.set_ylabel('PROGENy Pathway', fontsize=12)
ax.set_xlabel('Cell Type', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.figtext(
    0.02,
    -0.1,
    (
        'Figure. Pathway-level RNA-RPPA shift heatmap derived from the downstream inferential layer. '
        'Color values are display-normalized to [-1, 1] for visual contrast only; stars are based on '
        'the original permutation/FDR significance testing. Asterisks denote pathway-cell type tests '
        f'passing significance (* q < {SIG_Q_CUTOFF:.2f}, ** q < 0.01, *** q < 0.001).'
    ),
    wrap=True,
    horizontalalignment='left',
    fontsize=11,
)

plt.tight_layout()
output_path_avg = RESULTS_DIR / 'Average_Pathway_RPPA_Heatmap_nb14.svg'
plt.savefig(output_path_avg, format='svg', bbox_inches='tight')
plt.show()

print(f'Heatmap generated and saved to: {output_path_avg}')
print(f'Heatmap scale: {scale_note}')

### Figure of cell type specific pathway–RPPA correlations and pathway activity

In [ ]:
# =========================================================
# 1. PREPARE SYNCED DATA (Cell types shared across panels)
# =========================================================
if 'split_heat' not in globals() or split_heat.empty:
    raise RuntimeError('split_heat must be prepared before building the integrated figure.')
if 'star_annot' not in globals():
    raise RuntimeError('star_annot must be prepared before building the integrated figure.')

# Keep panel order consistent with the pathway summary block
cell_type_order = [ct for ct in split_heat.columns if ct in set(corr_df['cell_type'].unique())]
if len(cell_type_order) == 0:
    raise RuntimeError('No overlapping cell types between corr_df and split_heat.')

group_order = ['Mock', 'OG']
corr_df_filtered = corr_df[corr_df['cell_type'].isin(cell_type_order)].copy()
heat_filtered_raw = split_heat.reindex(columns=cell_type_order).copy()
heat_stars = star_annot.reindex(index=heat_filtered_raw.index, columns=cell_type_order).fillna('')

# Visual-only stretch to [0, 1] for non-directional shift magnitude
_hvals = heat_filtered_raw.to_numpy(dtype=float)
if not np.isfinite(_hvals).any():
    raise RuntimeError('No finite values found in split_heat for integrated figure panel B.')

_hmin = float(np.nanmin(_hvals))
_hmax = float(np.nanmax(_hvals))
if _hmax > _hmin:
    heat_filtered = (heat_filtered_raw - _hmin) / (_hmax - _hmin)
    heat_scale_note = f'display-normalized from raw range [{_hmin:.4f}, {_hmax:.4f}] to [0, 1]'
else:
    heat_filtered = pd.DataFrame(0.0, index=heat_filtered_raw.index, columns=heat_filtered_raw.columns)
    heat_scale_note = f'raw range is constant at {_hmin:.4f}; plotted as 0.0'

print(f"Sync complete. Plotting {len(cell_type_order)} aligned cell types.")

# =========================================================
# 2. ADJUSTMENT PANEL: Layout Configuration
# =========================================================
TOP_X, TOP_Y = 0.10, 0.65
TOP_WIDTH, TOP_HEIGHT = 0.80, 0.20
GAP = 0.02
Y_LIMIT_BUFFER = 0.15
ARROW_LENGTH, ARROW_WIDTH = 0.08, 1.0
BRACKET_WIDTH, BRACKET_Y_PAD, STAR_Y_PAD = 0.20, 0.03, 0.005

BTM_WIDTH, BTM_HEIGHT = TOP_WIDTH, 0.35
BTM_X, BTM_Y = TOP_X, TOP_Y - BTM_HEIGHT - GAP

# =========================================================
# 3. INITIALIZE CANVAS
# =========================================================
fig = plt.figure(figsize=(18, 14))
ax1 = fig.add_axes([TOP_X, TOP_Y, TOP_WIDTH, TOP_HEIGHT])
ax2 = fig.add_axes([BTM_X, BTM_Y, BTM_WIDTH, BTM_HEIGHT])

# --- PANEL A: Global Boxplot ---
sns.boxplot(
    x='cell_type', y='correlation', hue='group', data=corr_df_filtered,
    order=cell_type_order, hue_order=group_order, palette='Set2', showfliers=False, ax=ax1
)

sns.stripplot(
    x='cell_type', y='correlation', hue='group', data=corr_df_filtered,
    order=cell_type_order, hue_order=group_order, dodge=True, color='0.3', alpha=0.5, ax=ax1, legend=False
)

ax1.legend(loc='upper right', frameon=True, facecolor='white', framealpha=1.0)
ax1.set_xlim(-0.5, len(cell_type_order) - 0.5)
ax1.text(-0.07, 1.1, 'a.', transform=ax1.transAxes, fontsize=16, fontweight='normal', va='top')

# --- STATISTICAL ANNOTATIONS (Welch's t-test) ---
for i, ct in enumerate(cell_type_order):
    m_vals = corr_df_filtered[(corr_df_filtered['cell_type'] == ct) & (corr_df_filtered['group'] == 'Mock')]['correlation']
    o_vals = corr_df_filtered[(corr_df_filtered['cell_type'] == ct) & (corr_df_filtered['group'] == 'OG')]['correlation']

    if len(m_vals) > 1 and len(o_vals) > 1:
        _, pval = ttest_ind(m_vals, o_vals, equal_var=False)
        if pval < 0.05:
            stars = '****' if pval < 0.0001 else '***' if pval < 0.001 else '**' if pval < 0.01 else '*'
            y_max = corr_df_filtered[corr_df_filtered['cell_type'] == ct]['correlation'].max()
            bracket_y = y_max + BRACKET_Y_PAD
            ax1.plot([i - BRACKET_WIDTH, i + BRACKET_WIDTH], [bracket_y, bracket_y], color='red', lw=1.2)
            ax1.text(i, bracket_y + STAR_Y_PAD, stars, ha='center', va='bottom', color='red', fontweight='bold', fontsize=12)

_a_y_top = corr_df_filtered['correlation'].max()
_a_y_bot = min(0.0, corr_df_filtered['correlation'].min() - 0.05)
ax1.set_ylim(_a_y_bot, _a_y_top + Y_LIMIT_BUFFER)
ax1.set_title('Global RNA-Protein Coupling vs. Pathway Breakdown', fontsize=18, pad=20)
ax1.set_ylabel('Spearman R', fontsize=14)
ax1.set_xticklabels([])
ax1.set_xlabel('')
ax1.grid(False)

# --- ALIGNMENT ARROWS ---
trans = transforms.blended_transform_factory(ax1.transData, ax1.transAxes)
for i in range(len(cell_type_order)):
    ax1.annotate('', xy=(i, -ARROW_LENGTH), xytext=(i, 0), xycoords=trans,
                 arrowprops=dict(arrowstyle='->', lw=ARROW_WIDTH, color='black', shrinkA=0, shrinkB=0), clip_on=False)

# --- PANEL B: Heatmap (display-normalized non-directional shift magnitude) ---
ax2.text(-0.07, 1.05, 'b.', transform=ax2.transAxes, fontsize=16, fontweight='normal', va='top')
cbar_ax = fig.add_axes([BTM_X + BTM_WIDTH + 0.01, BTM_Y, 0.01, BTM_HEIGHT])

_n_cols = max(1, len(cell_type_order))
sns.heatmap(
    heat_filtered,
    cmap='Reds',
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor='white',
    annot=heat_stars,
    fmt='',
    annot_kws={'fontsize': max(6, 10 - _n_cols // 5)},
    ax=ax2,
    cbar_ax=cbar_ax,
    cbar_kws={'label': 'Display-normalized pathway shift magnitude (scaled to [0, 1])'},
)

ax2.set_ylabel('PROGENy Pathway', fontsize=14)
ax2.set_xlabel('Cell Type', fontsize=14)
ax2.set_xticks([x + 0.5 for x in range(len(cell_type_order))])
ax2.set_xticklabels(cell_type_order, rotation=45, ha='right', fontsize=10)

# =========================================================
# 4. CAPTION AND EXPORT
# =========================================================
CHAR_LIMIT = 145

description = (
    'Figure 1. Integrated multi-omic view of global RNA-RPPA coupling and pathway-level shift structure. '
    '(a) Global Regulatory Coupling: Spearman rank correlation (R) between RNA pseudobulk profiles and '
    'treatment-matched RPPA replicates by cell type, with Welch\'s t-test annotations. '
    '(b) Signaling Circuit Breakdown: heatmap of non-directional pathway shift magnitude (delta_r_rms), '
    'display-normalized to [0, 1] for contrast. A significant shift means the observed pathway-cell type '
    'delta_r_rms is larger than expected under permutation of Mock/OG labels and remains significant after '
    f'FDR correction (* q < {SIG_Q_CUTOFF:.2f}, ** q < 0.01, *** q < 0.001). '
    'Arrows align cell types across global and pathway-level panels.'
)

wrapped_text = textwrap.fill(description, width=CHAR_LIMIT)

plt.figtext(0.05, 0.01, wrapped_text,
            wrap=True,
            ha='left',
            fontsize=16,
            linespacing=1.5,
            fontweight='normal')

output_path = RESULTS_DIR / 'Figure_Integrated_Correlation_nb14.svg'
plt.savefig(output_path, format='svg', bbox_inches='tight', dpi=300)
plt.show()

print(f'Final figure with detailed caption exported to: {output_path}')
print(f'Panel B heatmap scale: {heat_scale_note}')

### Delta R Pathway-RPPA-Celltype by Group

In [ ]:

def plot_delta_signaling_heatmap(direction_matrix, star_matrix=None):
    """
    Plot directional pathway shift (OG - Mock) using a signed matrix.
    Positive values indicate tighter RNA-protein coupling in OG; negative indicates decoupling.
    """
    if direction_matrix is None or direction_matrix.empty:
        raise RuntimeError("direction_matrix is required and must be non-empty.")

    _raw = direction_matrix.copy()
    _stars = None
    if star_matrix is not None:
        _stars = star_matrix.reindex(index=_raw.index, columns=_raw.columns).fillna('')

    _vals = _raw.to_numpy(dtype=float)
    if not np.isfinite(_vals).any():
        raise RuntimeError("No finite values found in directional pathway shift matrix.")

    _abs_max = float(np.nanmax(np.abs(_vals)))
    if not np.isfinite(_abs_max) or _abs_max == 0:
        _abs_max = 1e-9
        _scale_note = "all directional shifts are ~0; using a tiny symmetric display range around zero"
    else:
        _scale_note = f"signed range [{np.nanmin(_vals):.4f}, {np.nanmax(_vals):.4f}] with symmetric limits +/-{_abs_max:.4f}"

    plt.figure(figsize=(max(16, _raw.shape[1] * 0.6), 9))
    sns.heatmap(
        _raw,
        cmap="vlag",
        center=0,
        vmin=-_abs_max,
        vmax=_abs_max,
        linewidths=0.5,
        linecolor='white',
        annot=_stars if _stars is not None else False,
        fmt='' if _stars is not None else '.2f',
        cbar_kws={'label': 'Directional pathway shift (delta_r_pathway; OG - Mock)'}
    )

    plt.title("Directional Pathway Shift (OG - Mock)", fontsize=16, pad=20)
    plt.ylabel("PROGENy Pathway", fontsize=12)
    plt.xlabel("Anatomical Cell Population", fontsize=12)
    plt.xticks(rotation=45, ha='right')

    _caption = (
        "Figure. Directional pathway-shift heatmap from permutation-aware statistics. "
        "Positive (red) values indicate increased RNA-protein coupling in OG vs Mock; "
        "negative (blue) values indicate reduced coupling (decoupling). "
        "Asterisks mark pathway-cell type pairs with FDR-significant shifts "
        f"(* q < {SIG_Q_CUTOFF:.2f}, ** q < 0.01, *** q < 0.001)."
    )
    plt.figtext(0.5, -0.05, _caption, wrap=True, horizontalalignment='center', fontsize=11)

    plt.tight_layout()
    _out = RESULTS_DIR / 'Directional_Delta_Signaling_Heatmap_nb14.svg'
    plt.savefig(_out, format='svg', bbox_inches='tight')
    plt.show()

    print(f"Directional delta signaling heatmap exported to: {_out}")
    print(f"Heatmap scale: {_scale_note}")
    return _raw


# --- EXECUTION ---
if 'corr_table_stats' not in globals() or corr_table_stats is None or corr_table_stats.empty:
    raise RuntimeError("corr_table_stats is required before plotting directional delta signaling heatmap.")

_stats = corr_table_stats.copy()
_stats['delta_r_pathway'] = pd.to_numeric(_stats['delta_r_pathway'], errors='coerce')
_stats['q_value'] = pd.to_numeric(_stats['q_value'], errors='coerce')

# Align to upstream selected pathways/cell types when available
if 'split_heat' in globals() and split_heat is not None and not split_heat.empty:
    _path_order = split_heat.index.tolist()
    _ct_order = split_heat.columns.tolist()
else:
    _path_order = sorted(_stats['pathway'].dropna().unique().tolist())
    _ct_order = sorted(_stats['cell_type'].dropna().unique().tolist())

dir_matrix = _stats.pivot_table(
    index='pathway',
    columns='cell_type',
    values='delta_r_pathway',
    aggfunc='mean',
).reindex(index=_path_order, columns=_ct_order)

dir_stars = _stats.assign(
    stars=_stats['q_value'].apply(
        lambda q: '***' if pd.notna(q) and q < 0.001 else ('**' if pd.notna(q) and q < 0.01 else ('*' if pd.notna(q) and q < SIG_Q_CUTOFF else ''))
    )
).pivot_table(
    index='pathway',
    columns='cell_type',
    values='stars',
    aggfunc='first',
).reindex(index=_path_order, columns=_ct_order).fillna('')

df_delta_results = plot_delta_signaling_heatmap(dir_matrix, dir_stars)

### Figure: Group dx/dy Pathway–RPPA by cell

In [ ]:
# =========================================================
# 1. PREPARE DATA: DIRECTIONAL SHIFT & SYNC
# =========================================================
if 'corr_table_stats' not in globals() or corr_table_stats is None or corr_table_stats.empty:
    raise RuntimeError('corr_table_stats must be prepared before this integrated directional figure block.')

_stats = corr_table_stats.copy()
_stats['delta_r_pathway'] = pd.to_numeric(_stats['delta_r_pathway'], errors='coerce')
_stats['q_value'] = pd.to_numeric(_stats['q_value'], errors='coerce')

# Use all cell types in alphabetical order for consistency
shared_cell_types = sorted(_stats['cell_type'].dropna().unique().tolist())
if len(shared_cell_types) == 0:
    raise RuntimeError('No cell types available in corr_table_stats.')

group_order = ["Mock", "OG"]

# Panel B directional matrix (OG - Mock)
dir_matrix = _stats.pivot_table(
    index='pathway',
    columns='cell_type',
    values='delta_r_pathway',
    aggfunc='mean',
).reindex(columns=shared_cell_types)

dir_stars = _stats.assign(
    stars=_stats['q_value'].apply(
        lambda q: '***' if pd.notna(q) and q < 0.001 else ('**' if pd.notna(q) and q < 0.01 else ('*' if pd.notna(q) and q < SIG_Q_CUTOFF else ''))
    )
).pivot_table(
    index='pathway',
    columns='cell_type',
    values='stars',
    aggfunc='first',
).reindex(index=dir_matrix.index, columns=shared_cell_types).fillna('')

_vals = dir_matrix.to_numpy(dtype=float)
if not np.isfinite(_vals).any():
    raise RuntimeError('No finite values found in directional shift matrix for panel B.')

_abs_max = float(np.nanmax(np.abs(_vals)))
if not np.isfinite(_abs_max) or _abs_max == 0:
    _abs_max = 1e-9
    heat_scale_note = 'all directional shifts are ~0; using tiny symmetric limits around zero'
else:
    heat_scale_note = f'signed range [{np.nanmin(_vals):.4f}, {np.nanmax(_vals):.4f}] with symmetric limits +/-{_abs_max:.4f}'

# Panel A data
corr_df_filtered = corr_df[corr_df["cell_type"].isin(shared_cell_types)].copy()

print(f"Data synced. Plotting directional integrated figure for {len(shared_cell_types)} cell populations.")

# =========================================================
# 2. LAYOUT CONFIGURATION
# =========================================================
TOP_X, TOP_Y = 0.10, 0.65
TOP_WIDTH, TOP_HEIGHT = 0.80, 0.20
GAP = 0.02
Y_LIMIT_BUFFER = 0.15
ARROW_LENGTH, ARROW_WIDTH = 0.08, 1.0
BRACKET_WIDTH, BRACKET_Y_PAD, STAR_Y_PAD = 0.20, 0.03, 0.005

BTM_WIDTH, BTM_HEIGHT = TOP_WIDTH, 0.35
BTM_X, BTM_Y = TOP_X, TOP_Y - BTM_HEIGHT - GAP

# =========================================================
# 3. INITIALIZE CANVAS
# =========================================================
fig = plt.figure(figsize=(18, 14))
ax1 = fig.add_axes([TOP_X, TOP_Y, TOP_WIDTH, TOP_HEIGHT])
ax2 = fig.add_axes([BTM_X, BTM_Y, BTM_WIDTH, BTM_HEIGHT])

# --- PANEL A: Global Boxplot (Absolute R) ---
sns.boxplot(x="cell_type", y="correlation", hue="group", data=corr_df_filtered,
            order=shared_cell_types, hue_order=group_order, palette="Set2", showfliers=False, ax=ax1)

sns.stripplot(x="cell_type", y="correlation", hue="group", data=corr_df_filtered,
              order=shared_cell_types, hue_order=group_order, dodge=True, palette='dark:0.3', alpha=0.5, ax=ax1, legend=False)

ax1.legend(loc='upper right', frameon=True, facecolor='white')
ax1.set_xlim(-0.5, len(shared_cell_types) - 0.5)
ax1.text(-0.07, 1.1, "a.", transform=ax1.transAxes, fontsize=18, fontweight='bold', va='top')

# --- STATISTICAL ANNOTATIONS ---
for i, ct in enumerate(shared_cell_types):
    m_vals = corr_df_filtered[(corr_df_filtered["cell_type"] == ct) & (corr_df_filtered["group"] == "Mock")]["correlation"]
    o_vals = corr_df_filtered[(corr_df_filtered["cell_type"] == ct) & (corr_df_filtered["group"] == "OG")]["correlation"]

    if len(m_vals) > 1 and len(o_vals) > 1:
        _, pval = ttest_ind(m_vals, o_vals, equal_var=False)
        if pval < 0.05:
            stars = "****" if pval < 0.0001 else "***" if pval < 0.001 else "**" if pval < 0.01 else "*"
            y_max = corr_df_filtered[corr_df_filtered["cell_type"] == ct]["correlation"].max()
            bracket_y = y_max + BRACKET_Y_PAD
            ax1.plot([i - BRACKET_WIDTH, i + BRACKET_WIDTH], [bracket_y, bracket_y], color='red', lw=1.2)
            ax1.text(i, bracket_y + STAR_Y_PAD, stars, ha='center', va='bottom', color='red', fontweight='bold', fontsize=12)

ax1.set_ylim(0, corr_df_filtered["correlation"].max() + Y_LIMIT_BUFFER)
ax1.set_title("Treatment-Induced Regulatory Shifts", fontsize=20, pad=25)
ax1.set_ylabel("Spearman R", fontsize=14)
ax1.set_xticklabels([])
ax1.set_xlabel("")

# --- ALIGNMENT ARROWS ---
trans = transforms.blended_transform_factory(ax1.transData, ax1.transAxes)
for i in range(len(shared_cell_types)):
    ax1.annotate('', xy=(i, -ARROW_LENGTH), xytext=(i, 0), xycoords=trans,
                 arrowprops=dict(arrowstyle="->", lw=ARROW_WIDTH, color='black', shrinkA=0, shrinkB=0), clip_on=False)

# --- PANEL B: Directional Shift Heatmap ---
ax2.text(-0.07, 1.05, "b.", transform=ax2.transAxes, fontsize=18, fontweight='bold', va='top')
cbar_ax = fig.add_axes([BTM_X + BTM_WIDTH + 0.01, BTM_Y, 0.01, BTM_HEIGHT])

_n_cols = max(1, len(shared_cell_types))
sns.heatmap(
    dir_matrix,
    cmap="vlag",
    center=0,
    vmin=-_abs_max,
    vmax=_abs_max,
    linewidths=0.5,
    linecolor='white',
    annot=dir_stars,
    fmt='',
    annot_kws={'fontsize': max(6, 10 - _n_cols // 5)},
    ax=ax2,
    cbar_ax=cbar_ax,
    cbar_kws={'label': 'Directional pathway shift (delta_r_pathway; OG - Mock)'}
)

ax2.set_ylabel("PROGENy Pathway", fontsize=14)
ax2.set_xlabel("Anatomical Cell Population", fontsize=14)
ax2.set_xticks([x + 0.5 for x in range(len(shared_cell_types))])
ax2.set_xticklabels(shared_cell_types, rotation=45, ha='right', fontsize=10)

# =========================================================
# 4. CAPTION AND EXPORT
# =========================================================
description = (
    "Figure 1. Treatment-induced remodeling of RNA-protein regulatory coupling. "
    "(a) Global coupling distribution: Spearman correlation ($R_s$) between RNA and RPPA replicates; "
    "Welch's t-test indicates significant Mock-vs-OG differences (*p < 0.05). "
    "(b) Directional signaling shift (delta_r_pathway = OG - Mock): red denotes increased coupling in OG, "
    "blue denotes reduced coupling (decoupling). Asterisks denote pathway-cell type pairs that remain "
    f"significant after FDR correction (* q < {SIG_Q_CUTOFF:.2f}, ** q < 0.01, *** q < 0.001). "
    "Arrows align global coupling changes with pathway-level directional shifts."
)

wrapped_text = textwrap.fill(description, width=140)
plt.figtext(0.05, 0.05, wrapped_text, wrap=True, ha='left', fontsize=16, linespacing=1.4)

output_path = RESULTS_DIR / 'Figure_Integrated_Directional_Shift_Correlation_nb14.svg'
plt.savefig(output_path, format='svg', bbox_inches='tight', dpi=300)
plt.show()

print(f"Unified directional shift figure exported to: {output_path}")
print(f"Panel B heatmap scale: {heat_scale_note}")

#### DF: corr_SamplePath

## Lead Protein Driver Analysis

In [ ]:
# Lead-driver setup (directional analysis-ready)

# 1. Define shared proteins from harmonized features when available
if 'shared_features' in globals() and shared_features is not None and len(shared_features) > 0:
    if isinstance(shared_features, pd.MultiIndex):
        _sf = shared_features.get_level_values(-1).astype(str).tolist()
    elif isinstance(shared_features, (pd.Index, pd.Series)):
        _sf = shared_features.astype(str).tolist()
    else:
        _sf = [str(x[-1] if isinstance(x, tuple) else x) for x in list(shared_features)]
    shared_proteins = sorted(set(_sf))
else:
    # Fallback: use proteins present in df_log and corr_table feature labels
    if 'corr_table' in globals() and 'feature' in corr_table.columns:
        _corr_feats = pd.Index(corr_table['feature'].astype(str).unique())
        _corr_prot = _corr_feats.str.split(' | ').str[-1]
        shared_proteins = sorted(list(set(df_log.columns.astype(str)) & set(_corr_prot)))
    else:
        shared_proteins = sorted(df_log.columns.astype(str).tolist())

# 2. Define pathways (prefer significance-aware pathway set if available)
if 'split_heat' in globals() and split_heat is not None and not split_heat.empty:
    pathways = split_heat.index.tolist()
else:
    pathways = [col for col in pseudobulk.columns if col not in ['cell_type', 'group']]

# 3. Define all cell types (alphabetical, consistent with upstream blocks)
if 'split_heat' in globals() and split_heat is not None and not split_heat.empty:
    all_cell_types = split_heat.columns.tolist()
else:
    all_cell_types = sorted(pb_meta['cell_type'].dropna().unique().tolist())

print('Setup complete:')
print(f'Pathways: {len(pathways)} | Proteins: {len(shared_proteins)} | Cell types: {len(all_cell_types)}')

Note: next block requires some computing power

In [ ]:
# 1. Initialize GPU/CPU device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Computing directional pathway-protein driver table on {device.type.upper()}...')

# 2. Build pathway-level table from the best available source
if 'pathway_scores' in globals() and isinstance(pathway_scores, pd.DataFrame):
    _ps = pathway_scores.copy()
    required = {'cell_type', 'group'}
    if not required.issubset(_ps.columns):
        raise RuntimeError('pathway_scores must contain cell_type and group columns for directional driver analysis.')

    # Candidate pathway columns are numeric columns not part of metadata
    meta_cols = {'cell_type', 'group', 'sample', 'base_sample'}
    path_cols = [c for c in _ps.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(_ps[c])]
    if not path_cols:
        raise RuntimeError('No numeric pathway columns found in pathway_scores.')

    pathway_table = _ps[['cell_type', 'group'] + path_cols].copy()
else:
    # Fallback to pseudobulk only if pathway columns are truly present
    path_cols = [p for p in pathways if p in pseudobulk.columns]
    if not path_cols:
        raise RuntimeError('No pathway source available: neither pathway_scores numeric pathway columns nor matching pseudobulk columns found.')

    if 'cell_type' not in pseudobulk.columns or 'group' not in pseudobulk.columns:
        raise RuntimeError('pseudobulk must include cell_type and group columns for fallback pathway analysis.')

    pathway_table = pseudobulk[['cell_type', 'group'] + path_cols].copy()

# Keep path list aligned with actual table columns
pathways = path_cols

# 3. Precompute global protein shifts (OG - Mock)
mock_p = df_log[df_log.index.str.contains('Mock')][shared_proteins].mean(axis=0)
og_p = df_log[df_log.index.str.contains('OG')][shared_proteins].mean(axis=0)
protein_delta = (og_p - mock_p).astype(float)

# 4. Compute directional pathway-protein concordance proxy per cell type
rows = []
for ct in tqdm(all_cell_types):
    ct_tbl = pathway_table[pathway_table['cell_type'] == ct]
    if ct_tbl.empty:
        continue

    if (ct_tbl['group'] == 'Mock').sum() == 0 or (ct_tbl['group'] == 'OG').sum() == 0:
        continue

    rna_mock = ct_tbl.loc[ct_tbl['group'] == 'Mock', pathways].mean(axis=0)
    rna_og = ct_tbl.loc[ct_tbl['group'] == 'OG', pathways].mean(axis=0)
    rna_delta = (rna_og - rna_mock).astype(float)

    for pw in pathways:
        d_rna = float(rna_delta.get(pw, np.nan))
        if not np.isfinite(d_rna):
            continue
        for prot in shared_proteins:
            d_prot = float(protein_delta.get(prot, np.nan))
            if not np.isfinite(d_prot):
                continue

            # Signed concordance proxy: positive when RNA/protein shifts align
            delta_r = d_rna * d_prot
            rows.append({
                'cell_type': ct,
                'pathway': pw,
                'protein': prot,
                'delta_r': float(delta_r),
                'delta_rna': d_rna,
                'delta_protein': d_prot,
            })

master_driver_table = pd.DataFrame(rows)
if master_driver_table.empty:
    raise RuntimeError('No directional driver rows were generated; check pathway/protein setup and group coverage.')

print('Success! Directional driver table complete.')
print(f'Rows: {master_driver_table.shape[0]:,}')
print(f"Max |delta_r|: {master_driver_table['delta_r'].abs().max():.4g}")

### Cluster-agnostic Top 20 Degoupling Events

In [ ]:
# Global interaction specificity calculation (directional hotspots)

# 1. Re-format the entire dataset into a wide matrix (Interactions x Cell Types)
if 'interaction' not in master_driver_table.columns:
    master_driver_table['interaction'] = master_driver_table['pathway'] + " | " + master_driver_table['protein']

full_matrix = master_driver_table.pivot_table(index='interaction', columns='cell_type', values='delta_r', aggfunc='mean')

# 2. Calculate directional specificity index (row-wise Z-score across cell types)
row_means = full_matrix.mean(axis=1)
row_stds = full_matrix.std(axis=1)
z_matrix = full_matrix.sub(row_means, axis=0).div(row_stds + 1e-9, axis=0)

# 3. Score interactions by strongest absolute specificity and peak cluster
interaction_peak_abs = z_matrix.abs().max(axis=1)
interaction_peak_cluster = z_matrix.abs().idxmax(axis=1)

ranking_df = pd.DataFrame({
    'interaction': interaction_peak_abs.index,
    'peak_abs_z': interaction_peak_abs.values,
    'peak_cluster': interaction_peak_cluster.reindex(interaction_peak_abs.index).values,
})
ranking_df['pathway'] = ranking_df['interaction'].str.split(' | ').str[0]
ranking_df = ranking_df.sort_values('peak_abs_z', ascending=False)

# 4. Diversified selection to prevent one-axis saturation (e.g., Immune/JAK-STAT)
DISPLAY_INTERACTIONS = 30
MAX_PER_PATHWAY = 4
MAX_IMMUNE_PEAK = 2

selected = []
pathway_counts = {}
immune_peak_count = 0

for _, row in ranking_df.iterrows():
    interaction = row['interaction']
    pathway = row['pathway']
    peak_cluster = str(row['peak_cluster'])

    if pathway_counts.get(pathway, 0) >= MAX_PER_PATHWAY:
        continue

    is_immune_peak = ('Immune' in peak_cluster)
    if is_immune_peak and immune_peak_count >= MAX_IMMUNE_PEAK:
        continue

    selected.append(interaction)
    pathway_counts[pathway] = pathway_counts.get(pathway, 0) + 1
    if is_immune_peak:
        immune_peak_count += 1

    if len(selected) >= DISPLAY_INTERACTIONS:
        break

# Fallback fill if caps are too strict for current data
if len(selected) < DISPLAY_INTERACTIONS:
    fallback = ranking_df.loc[~ranking_df['interaction'].isin(selected), 'interaction'].tolist()
    selected.extend(fallback[: DISPLAY_INTERACTIONS - len(selected)])

top_agnostic_names = selected

# 5. Prepare plotting data
plot_matrix = full_matrix.loc[top_agnostic_names]
plot_z_matrix = z_matrix.loc[top_agnostic_names]

# 6. Export for downstream Phase 2 analysis
de_output = plot_matrix.copy()

top_name = top_agnostic_names[0]
print(f"Discovery complete. Ranked {len(full_matrix)} interactions across {full_matrix.shape[1]} clusters.")
print(f"Selected {len(top_agnostic_names)} interactions with caps: pathway<={MAX_PER_PATHWAY}, immune-peak<={MAX_IMMUNE_PEAK}")
print(f"Top directional hotspot: {top_name} with peak specificity Z={z_matrix.loc[top_name].loc[z_matrix.loc[top_name].abs().idxmax()]:.2f}")

In [ ]:
# --- STEP 1: PRE-ALIGN THE STRINGS ---
# This forces the "|" to line up by padding shorter names with spaces
def align_labels(labels):
    # Split into (Pathway, Protein)
    parts = [l.split(" | ") for l in labels]
    # Find the max length of the left side (Pathways)
    max_path_len = max(len(p[0]) for p in parts)
    # Rebuild strings with padding
    return [f"{p[0]:>{max_path_len}} | {p[1]}" for p in parts]

# Update the index of your plotting dataframe with the aligned strings
plot_z_matrix.index = align_labels(plot_z_matrix.index)

# --- STEP 2: THE PLOT ---
fig = plt.figure(figsize=(24, 16))

ax = sns.heatmap(
    plot_z_matrix,
    cmap="RdBu_r",
    center=0,
    vmin=-4,
    vmax=4,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Regional directional specificity (row-wise Z-score)', 'shrink': 0.7}
)

n_display = plot_z_matrix.shape[0]
plt.title(f"Cluster-Agnostic Discovery: Top {n_display} Diversified Directional Hotspots\n(Ranked by Peak |Specificity Z| with pathway/immune caps)", fontsize=22, pad=35)
plt.ylabel(f"Top {n_display} Directional Interaction Hotspots", fontsize=16, labelpad=40)

# Aligned Header
ax.annotate('RNA Pathway | Protein', xy=(-0.005, 1.02), xycoords='axes fraction',
            fontsize=14, fontweight='normal', ha='right', va='bottom')

# X-axis coloring
for label in ax.get_xticklabels():
    txt = label.get_text()
    if any(x in txt for x in ['Vascular', 'Immune', 'OEC', 'Astro-Epen']):
        label.set_color('red')
        label.set_weight('bold')
    elif 'MB' in txt or 'IT-ET' in txt:
        label.set_color('blue')
    else:
        label.set_color('black')

plt.xticks(rotation=45, ha='right', fontsize=12)

# APPLY MONOSPACE: This is required to make the padding work visually
plt.yticks(fontsize=12, family='monospace', ha='right')

# Legends
barrier_patch = mpatches.Patch(color='red', label='Barrier & Defense')
mb_patch = mpatches.Patch(color='blue', label='Midbrain & Cortical Neurons')

ax.legend(handles=[barrier_patch, mb_patch],
          loc='upper center', bbox_to_anchor=(0.5, -0.2),
          ncol=3, fontsize=18, frameon=False, title="Cluster Region Mapping", title_fontsize=16)

figure_legend_raw = (
    f"Figure X: Diversified unsupervised directional hotspot discovery. "
    f"All pathway-protein interactions were profiled across all available cell-type clusters using directional "
    f"delta_r scores (OG - Mock). A row-wise specificity index (Z-score across clusters) was computed to identify "
    f"interaction signals that are extreme outliers in specific regions rather than broad, non-specific shifts. "
    f"To reduce dominance by a single immune axis, ranking is constrained by pathway and immune-peak caps. "
    f"The heatmap shows the top {n_display} interactions after diversification. Red indicates OG-up directional "
    f"specificity in a given cluster; blue indicates OG-down directional specificity."
)

wrapped_legend = "\n".join(textwrap.wrap(figure_legend_raw, width=150))
fig.text(0.02, 0.12, wrapped_legend, ha='left', va='top', fontsize=20, fontweight='medium')

plt.tight_layout(rect=[0, 0.12, 1, 0.98])

# Save
save_path = RESULTS_DIR / "Figure_TopDiversified_Directional_Hotspot_Heatmap_nb14.svg"
plt.savefig(save_path, format="svg")

plt.show()
print(f"Directional hotspot figure exported to: {save_path}")

# Neuronal Pathway Protein Drivers Analysis

## Create Neuronal sorted df

In [ ]:
# Define the High-Biomass Neuronal Peer Group
# These are the clusters that define the "Global Baseline" for nerve cells
neuronal_peer_group = [
    '01 IT-ET Glut', '02 NP-CT-L6b Glut', '05 OB-IMN GABA', 
    '06 CTX-CGE GABA', '07 CTX-MGE GABA', '08 CNU-MGE GABA', 
    '10 LSX GABA', '11 CNU-HYa GABA', '12 HY GABA', 
    '13 CNU-HYa Glut', '18 TH Glut', '19 MB Glut', '20 MB GABA', '33 Vascular'
]

# Create the specific neuronal subset
neural_df = master_driver_table[master_driver_table['cell_type'].isin(neuronal_peer_group)].copy()

# Create the raw Delta R pivot
neural_pivot = neural_df.pivot(index='interaction', columns='cell_type', values='delta_r').fillna(0)

# Identify the Top 20 outliers specific to these neuronal clusters
neural_top_20 = neural_pivot.min(axis=1).sort_values().head(20).index

# Filter the pivot to only these top interactions
neural_plot_matrix = neural_pivot.loc[neural_top_20]

# PERMANENT NAME: neural_z_matrix
# This scales the signal relative to other neurons, not the whole brain
neural_z_matrix = neural_plot_matrix.apply(lambda x: (x - x.mean()) / (x.std() + 1e-9), axis=1)

## Plot

In [ ]:
# --- STEP 1: ALIGN INTERACTION LABELS ---
# Forces the "|" to line up for a professional, clean Y-axis
def align_neural_labels(labels):
    parts = [l.split(" | ") for l in labels]
    max_path_len = max(len(p[0]) for p in parts)
    return [f"{p[0]:>{max_path_len}} | {p[1]}" for p in parts]

neural_z_matrix.index = align_neural_labels(neural_z_matrix.index)

# --- STEP 2: THE PLOT ---
fig = plt.figure(figsize=(24, 16))

# Using your preferred +/- 3 scale for high-contrast neuronal specificity
ax = sns.heatmap(neural_z_matrix, cmap="RdBu_r", center=0, vmin=-3, vmax=3,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Regional Specificity Index (Z-Score)', 'shrink': 0.8})

# Title and Axis Labels
plt.title("Regional Specificity: High-Biomass Neuronal Outliers\n(Unsupervised Discovery of Subcortical Vulnerability)", 
          fontsize=24, pad=40, fontweight='normal')
plt.ylabel("Top 20 Decoupled Interactions (Ranked by Specificity)", fontsize=18, labelpad=45)

# RNA | Protein Header (Aligned to the Y-axis)
ax.annotate('RNA Pathway | Protein', xy=(-0.005, 1.02), xycoords='axes fraction', 
            fontsize=14, fontweight='normal', ha='right', va='bottom')

# --- STEP 3: X-AXIS STYLING ---
for label in ax.get_xticklabels():
    txt = label.get_text()
    if '10 LSX' in txt:
        label.set_color('navy')
        label.set_fontweight('bold')
        label.set_fontsize(14)
    elif 'MB' in txt:
        label.set_color('blue')
        label.set_fontweight('bold')
    elif '33 Vascular' in txt:
        label.set_color('darkred')
        label.set_fontweight('bold')
    else:
        label.set_alpha(0.5)

plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=13, family='monospace', ha='right') 

# --- STEP 4: LEGEND & CAPTION ---
# Creating the patches to match your previous figure style
lsx_patch = mpatches.Patch(color='navy', label='Lateral Septum (LSX)')
mb_patch = mpatches.Patch(color='blue', label='Midbrain (MB)')
vasc_patch = mpatches.Patch(color='darkred', label='Vascular Control')

ax.legend(handles=[lsx_patch, mb_patch, vasc_patch], 
          loc='upper center', bbox_to_anchor=(0.5, -0.15), 
          ncol=3, fontsize=16, frameon=False)

# Professional Figure Legend
figure_legend = (
    "Figure Y: Selective Neuronal Vulnerability in Subcortical Circuits. "
    "To isolate neuronal-specific impacts from global barrier effects, 13 neuronal clusters "
    "were analyzed alongside vascular controls. Heatmap values represent the Regional Specificity Index (Z-score), "
    "identifying interactions where the OG-driven decoupling (Delta R) is a statistical outlier relative to the "
    "neuronal baseline. Note the striking vertical 'blue strip' in the 10 LSX GABA cluster, indicating a "
    "localized collapse of Nectin4 and Esr1-mediated regulatory networks, a signature frequently associated "
    "with synaptic pruning and neurodegenerative vulnerability."
)

wrapped_legend = "\n".join(textwrap.wrap(figure_legend, width=170))
fig.text(0.02, 0.1, wrapped_legend, ha='left', va='top', fontsize=18, fontweight='light')

plt.tight_layout(rect=[0, 0.1, 1, 0.95])

# Save
save_path = RESULTS_DIR / "Figure_Neural_Specificity_Heatmap_nb14.svg"
plt.savefig(save_path, format="svg")

plt.show()
print(f"Neural specificity figure exported to: {save_path}")

# DoRothEA

## Master DoRothEA: tf_activity_master

In [ ]:
# =========================================================
# GFX MODULE: Unified DoRothEA Master (Robust ULM Version)
# =========================================================
logger.info("Computing DoRothEA TF activities using robust Univariate Linear Modeling...")

# 1. Build mouse-target DoRothEA net from human + local ortholog mapping.
# dc.op.dorothea(organism='mouse') may trigger remote HCOP fetches that can fail (404).
ortholog_df = pd.read_csv(ORTHOLOG_PATH)
h2m = {
    h: m
    for h, m in zip(ortholog_df["Human_Gene"], ortholog_df["Mouse_Gene"])
    if pd.notna(h) and pd.notna(m)
}

dorothea_human = dc.op.dorothea(organism="human", levels=["A", "B", "C"])
# Translate human targets to mouse symbols using local NB02 map
dorothea_net = dorothea_human.copy()
dorothea_net["target"] = dorothea_net["target"].map(h2m)
dorothea_net = (
    dorothea_net.dropna(subset=["target"])
.drop_duplicates(subset=["source", "target"], keep="first")
.reset_index(drop=True)
)

logger.info(
    f"DoRothEA mouse network built locally: {len(dorothea_net)} TF-target edges across "
    f"{dorothea_net['source'].nunique()} TFs"
)

# 2. Run enrichment (ULM - robust against singular matrix issues).
tf_scores, _ = dc.mt.ulm(data=expr_pb, net=dorothea_net)

# 3. Integrate metadata robustly (no dependency on a single upstream variable name).
meta_candidates = [
    globals().get("pb_meta"),
    globals().get("pseudobulk_meta"),
    globals().get("pb_meta_local"),
]
meta_source = next((m for m in meta_candidates if isinstance(m, pd.DataFrame)), None)
if meta_source is None:
    raise ValueError("No pseudobulk metadata table found (expected pb_meta/pseudobulk_meta/pb_meta_local).")

meta_source = meta_source.copy().reset_index(drop=True)
if "sample" not in meta_source.columns:
    raise ValueError("Metadata table is missing required column: 'sample'")
if "cell_type" not in meta_source.columns:
    raise ValueError("Metadata table is missing required column: 'cell_type'")

if len(meta_source) != len(tf_scores):
    raise ValueError(
        f"Metadata/sample length mismatch for TF scores: {len(meta_source)} metadata rows vs {len(tf_scores)} score rows"
    )

tf_scores["base_sample"] = meta_source["sample"].astype(str).values
tf_scores["cell_type"] = meta_source["cell_type"].astype(str).values
tf_scores["group"] = tf_scores["base_sample"].apply(lambda x: "Mock" if "Mock" in x else "OG")

# --- Create tf_activity_master (long format) ---
tf_cols = tf_scores.columns.drop(["base_sample", "cell_type", "group"]).tolist()
tf_activity_master = tf_scores.melt(
    id_vars=["base_sample", "cell_type", "group"],
    value_vars=tf_cols,
    var_name="tf_name",
    value_name="activity_score",
)

def _safe_sheet_name(name: str, used_names: set) -> str:
    sheet = str(name)
    for bad in ["[", "]", ":", "*", "?", "/", "\\"]:
        sheet = sheet.replace(bad, "_")
    sheet = (sheet.strip() or "Sheet")[:31]
    base = sheet
    i = 1
    while sheet in used_names:
        suffix = f"_{i}"
        sheet = f"{base[: 31 - len(suffix)]}{suffix}"
        i += 1
    used_names.add(sheet)
    return sheet

# Save for downstream plotting (long-format CSV + per-cell-type Excel tabs)
activity_out = RESULTS_DIR / "Sample_Level_TF_Activities_nb14.csv"
activity_xlsx_out = RESULTS_DIR / "Sample_Level_TF_Activities_ByCell_nb14.xlsx"
tf_activity_master.to_csv(activity_out, index=False)

activity_sheet_map = []
with pd.ExcelWriter(activity_xlsx_out, engine="openpyxl") as writer:
    used_names = set()
    for ct, sub in tf_activity_master.groupby("cell_type", observed=True):
        sheet = _safe_sheet_name(ct, used_names)
        sub.sort_values(["group", "base_sample", "tf_name"]).to_excel(writer, sheet_name=sheet, index=False)
        activity_sheet_map.append({"cell_type": ct, "sheet_name": sheet})
    pd.DataFrame(activity_sheet_map).to_excel(writer, sheet_name="_sheet_map", index=False)

logger.info(f"Created tf_activity_master with {len(tf_cols)} TFs.")
logger.info(f"Saved sample-level TF activity table to {activity_out}")
logger.info(f"Saved sample-level TF activity workbook (tabs per cell type) to {activity_xlsx_out}")

# =========================================================
# 4. Parallel TF-RPPA correlations (using same helper as gene/pathway branch)
# =========================================================
tf_group_tasks = list(tf_scores.groupby(["cell_type", "group"], observed=True))
logger.info(f"Starting parallel TF correlation for {len(tf_cols)} TFs across {len(tf_group_tasks)} groups.")

results_lists_tf = Parallel(n_jobs=N_JOBS)(
    delayed(compute_ct_correlations)(
        ct,
        grp,
        sub,
        df_log,
        shared_proteins,
        tf_cols,
        rppa_group_map,
        MIN_RPPA_REPS_PER_GROUP,
    )
    for (ct, grp), sub in tqdm(tf_group_tasks, desc="Processing TFs")
)

tf_corr_table = pd.DataFrame([item for sublist in results_lists_tf for item in sublist])

# 5. Export results (long-format CSV + per-cell-type Excel tabs)
tf_output_path = RESULTS_DIR / "Master_TF_RPPA_Correlations_nb14.csv"
tf_output_xlsx_path = RESULTS_DIR / "Master_TF_RPPA_Correlations_ByCell_nb14.xlsx"
tf_corr_table.to_csv(tf_output_path, index=False)

if "cell_type" not in tf_corr_table.columns:
    tf_corr_table["cell_type"] = "all"

corr_sheet_map = []
with pd.ExcelWriter(tf_output_xlsx_path, engine="openpyxl") as writer:
    used_names = set()
    if tf_corr_table.empty:
        pd.DataFrame([{"note": "No TF-RPPA correlations generated."}]).to_excel(
            writer, sheet_name="_no_data", index=False
        )
    else:
        for ct, sub in tf_corr_table.groupby("cell_type", observed=True):
            sheet = _safe_sheet_name(ct, used_names)
            sort_cols = [
                c for c in ["group", "tf", "tf_name", "pathway", "protein", "feature"] if c in sub.columns
            ]
            sub_export = sub.sort_values(sort_cols) if sort_cols else sub.copy()
            sub_export.to_excel(writer, sheet_name=sheet, index=False)
            corr_sheet_map.append({"cell_type": ct, "sheet_name": sheet})
    pd.DataFrame(corr_sheet_map).to_excel(writer, sheet_name="_sheet_map", index=False)

logger.info(
    "DoRothEA GFX pipeline complete. Results saved to "
    f"{tf_output_path} and {tf_output_xlsx_path}"
)

## Analysis and Figures

## Focus on significance-selected cluster TFs

In [ ]:
# =========================================================
# GFX: Significance-driven target discovery (Mock vs OG)
# Essential outputs only: one GWAS-ready CSV + in-memory stats table
# =========================================================
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# Thresholds for dynamic target selection
TF_MIN_REPS = 2
TF_P_CUTOFF = 0.05
TF_Q_CUTOFF = 0.10
TOP_TARGET_CLUSTERS = 3

# 1. Candidate clusters with enough replicates in both groups
cluster_counts = (
    tf_activity_master.groupby(['cell_type', 'group'])['activity_score']
    .size()
    .unstack(fill_value=0)
)
eligible_clusters = cluster_counts[
    (cluster_counts.get('Mock', 0) >= TF_MIN_REPS) &
    (cluster_counts.get('OG', 0) >= TF_MIN_REPS)
]
eligible_clusters = eligible_clusters.index.tolist()

# 2. Per-cluster/per-TF significance tests (single pass)
regional_stats = []
for ct in eligible_clusters:
    ct_data = tf_activity_master[tf_activity_master['cell_type'] == ct]
    for tf in sorted(ct_data['tf_name'].astype(str).unique()):
        mock_vals = ct_data[(ct_data['tf_name'] == tf) & (ct_data['group'] == 'Mock')]['activity_score']
        og_vals = ct_data[(ct_data['tf_name'] == tf) & (ct_data['group'] == 'OG')]['activity_score']
        if len(mock_vals) >= TF_MIN_REPS and len(og_vals) >= TF_MIN_REPS:
            delta = float(og_vals.mean() - mock_vals.mean())
            _, pval = ttest_ind(og_vals, mock_vals, nan_policy='omit')
            regional_stats.append({
                'tf_name': tf,
                'cell_type': ct,
                'delta': delta,
                'p_value': float(pval),
            })

stats_results_df = pd.DataFrame(regional_stats)
if stats_results_df.empty:
    raise ValueError('No eligible TF statistics computed. Check group/sample coverage.')

# 3. Multiple-testing correction and GWAS-ready derived columns
stats_results_df['q_value'] = multipletests(
    stats_results_df['p_value'].fillna(1.0), method='fdr_bh'
)[1]
stats_results_df['abs_delta'] = stats_results_df['delta'].abs()
stats_results_df['direction'] = stats_results_df['delta'].map(lambda x: 'OG_up' if x > 0 else ('Mock_up' if x < 0 else 'no_change'))

# Rank TFs within each cell type for immediate downstream selection
stats_results_df['rank_within_cell_type'] = (
    stats_results_df
    .sort_values(['cell_type', 'q_value', 'abs_delta', 'p_value'], ascending=[True, True, False, True], na_position='last')
    .groupby('cell_type')
    .cumcount() + 1
)

# Per-TF global support fields for GWAS triage
tf_global = (
    stats_results_df.groupby('tf_name', as_index=False)
    .agg(
        best_q=('q_value', 'min'),
        best_p=('p_value', 'min'),
        max_abs_delta=('abs_delta', 'max'),
        n_celltypes_tested=('cell_type', 'nunique'),
        n_celltypes_q10=('q_value', lambda s: int((s < 0.10).sum())),
        n_celltypes_q05=('q_value', lambda s: int((s < 0.05).sum())),
    )
)

stats_results_df = stats_results_df.merge(tf_global, on='tf_name', how='left')
stats_results_df = stats_results_df.sort_values(
    ['cell_type', 'q_value', 'abs_delta', 'p_value'],
    ascending=[True, True, False, True],
    na_position='last',
).reset_index(drop=True)

# 4. Significance-driven focus cluster (used by neighboring summary cells)
sig_counts = (
    stats_results_df[stats_results_df['q_value'] < TF_Q_CUTOFF]
    .groupby('cell_type')['tf_name']
    .nunique()
    .sort_values(ascending=False)
)

selected_target_clusters = sig_counts.index.tolist()[:TOP_TARGET_CLUSTERS]
if len(selected_target_clusters) == 0:
    selected_target_clusters = (
        stats_results_df.groupby('cell_type')['abs_delta']
        .median()
        .sort_values(ascending=False)
        .index.tolist()[:TOP_TARGET_CLUSTERS]
    )

focus_cluster = selected_target_clusters[0] if selected_target_clusters else None
focus_sig_df = stats_results_df[stats_results_df['cell_type'] == focus_cluster].copy() if focus_cluster else pd.DataFrame()

# 5. Essential export only: GWAS-ready all-cell TF table
stats_csv_out = RESULTS_DIR / 'TF_Significance_AllCells_nb14.csv'
stats_results_df.to_csv(stats_csv_out, index=False)

if focus_cluster is not None:
    focus_hits = focus_sig_df[focus_sig_df['q_value'] < TF_Q_CUTOFF]
    print(f"Significance-driven focus cluster: {focus_cluster}")
    print(f"Significant TFs in focus cluster (q < {TF_Q_CUTOFF:.2f}): {len(focus_hits)}")
else:
    print('No focus cluster identified from significance discovery.')

print(f"Saved GWAS-ready TF significance CSV to: {stats_csv_out}")
print(f"Rows: {len(stats_results_df):,} | TFs: {stats_results_df['tf_name'].nunique():,} | Cell types: {stats_results_df['cell_type'].nunique():,}")

In [ ]:
# Lightweight post-significance summary (no recomputation).
# Keeps convenience variables used for interpretation while avoiding duplicate work.

if not isinstance(globals().get('stats_results_df', None), pd.DataFrame) or stats_results_df.empty:
    raise ValueError('stats_results_df is missing. Run the previous significance cell first.')

TF_MIN_REPS = globals().get('TF_MIN_REPS', 2)
TF_P_CUTOFF = globals().get('TF_P_CUTOFF', 0.05)
MAX_NEIGHBORHOOD = 4
MAX_ANCHOR_TFS = 16

# Use existing significance table to pick neighborhood clusters.
sig_counts_p = (
    stats_results_df[stats_results_df['p_value'] < TF_P_CUTOFF]
    .groupby('cell_type')['tf_name']
    .nunique()
    .sort_values(ascending=False)
)

neighborhood = sig_counts_p.index.tolist()[:MAX_NEIGHBORHOOD]
if len(neighborhood) < MAX_NEIGHBORHOOD:
    extra = (
        stats_results_df.groupby('cell_type')['abs_delta']
        .median()
        .sort_values(ascending=False)
        .index.tolist()
    )
    for ct in extra:
        if ct not in neighborhood:
            neighborhood.append(ct)
        if len(neighborhood) >= MAX_NEIGHBORHOOD:
            break

# Pick anchor TFs from existing stats (significance first, then effect-size fallback).
significant_tfs = (
    stats_results_df[stats_results_df['p_value'] < TF_P_CUTOFF]
    .groupby('tf_name')['abs_delta']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
anchor_tfs = significant_tfs[:MAX_ANCHOR_TFS]
if len(anchor_tfs) == 0:
    anchor_tfs = (
        stats_results_df.groupby('tf_name')['abs_delta']
        .mean()
        .sort_values(ascending=False)
        .head(MAX_ANCHOR_TFS)
        .index.tolist()
    )

focus_cluster = neighborhood[0] if len(neighborhood) else None
n_sig = int((stats_results_df['p_value'] < TF_P_CUTOFF).sum())

print('Post-DoRothEA summary (reusing existing stats table):')
print(f"  P-threshold: p < {TF_P_CUTOFF:.3f}")
print(f"  Neighborhood clusters ({len(neighborhood)}): {neighborhood}")
print(f"  Anchor TFs ({len(anchor_tfs)}): {anchor_tfs[:8]}{' ...' if len(anchor_tfs) > 8 else ''}")
print(f"  Significant tests: {n_sig} / {len(stats_results_df)}")
if n_sig == 0:
    print('  No significant TF/cluster pairs under p-threshold; effect-size ordering is active.')

Significance-selected focus cluster and neighborhood are computed dynamically from TF tests.

Cluster inclusion criteria:
- Minimum replicates in both groups
- Prioritize clusters by number of TF hits passing FDR threshold
- Fallback to effect-size ranking when no FDR-significant hits are detected

TF inclusion criteria:
- Prioritize TFs passing global FDR threshold across tested cluster-TF pairs
- Fallback to highest mean absolute delta activity if no FDR-significant TFs

In [ ]:
# 1. Use all cell types and TFs (not restricted to dynamic neighborhood/anchors).
plot_df = stats_results_df.copy()

all_cell_types_sorted = sorted(plot_df["cell_type"].unique())
all_tfs_sorted = sorted(plot_df["tf_name"].unique())

heat_df = (plot_df.pivot(index="tf_name", columns="cell_type", values="delta")
           .reindex(all_tfs_sorted)
           .reindex(columns=all_cell_types_sorted))
pvals_df = (plot_df.pivot(index="tf_name", columns="cell_type", values="p_value")
            .reindex(all_tfs_sorted)
            .reindex(columns=all_cell_types_sorted))

n_stars = int((pvals_df < TF_P_CUTOFF).sum().sum())
n_strong = int((pvals_df < 0.01).sum().sum())

# 2. Plotting
fig_height = max(24, 0.42 * len(all_tfs_sorted))
fig, ax = plt.subplots(figsize=(16, fig_height))
sns.heatmap(
    heat_df,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "Delta Activity (OG - Mock)"},
)

# Overlay significance stars as markers so they sit at the exact center of each tile.
weak_y, weak_x = np.where((pvals_df < TF_P_CUTOFF) & ~(pvals_df < 0.01))
strong_y, strong_x = np.where(pvals_df < 0.01)

if len(weak_x):
    ax.scatter(
        weak_x + 0.5,
        weak_y + 0.5,
        marker="*",
        s=28,
        c="black",
        linewidths=0,
        zorder=5,
        clip_on=False,
    )

if len(strong_x):
    strong_x_center = strong_x + 0.5
    strong_y_center = strong_y + 0.5
    strong_offset = 0.13
    ax.scatter(
        strong_x_center - strong_offset,
        strong_y_center,
        marker="*",
        s=28,
        c="black",
        linewidths=0,
        zorder=6,
        clip_on=False,
    )
    ax.scatter(
        strong_x_center + strong_offset,
        strong_y_center,
        marker="*",
        s=28,
        c="black",
        linewidths=0,
        zorder=6,
        clip_on=False,
    )

ax.set_title(
    f"Full TF Activity Map Across All Cell Types\n(* p < {TF_P_CUTOFF:.3f}, ** p < 0.010)",
    fontsize=18,
    fontweight="bold"
    )
ax.set_ylabel("Transcription Factors", fontsize=12)
ax.set_xlabel("Cell Type Clusters", fontsize=12)
fig.tight_layout()

# 3. Save as SVG for publication editing
full_tf_map_out = RESULTS_DIR / "Full_TF_Significance_Map_AllCells_nb14.svg"
fig.savefig(full_tf_map_out)
plt.show()
print(f"Full TF significance map (all cells) exported to: {full_tf_map_out}")
print(f"Starred cells (p < {TF_P_CUTOFF:.3f}): {n_stars}")
print(f"Strong stars (p < 0.010): {n_strong}")